## Initial Code

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
df_aapl = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/stocks/AAPL.csv')

In [ ]:
import numpy as np
from scipy.stats import boxcox

df_aapl['Close_log'] = np.log(df_aapl['Close'] + 1)
df_aapl['Close_sqrt'] = np.sqrt(df_aapl['Close'])
df_aapl['Close_boxcox'], _ = boxcox(df_aapl['Close'] + 1)


This code calculates the skewness of the 'Close' column in the `df_aapl` DataFrame before and after applying various transformations:

1. **Original Skewness**: Calculates the skewness of the original 'Close' data.
2. **Log Transformation Skewness**: Calculates the skewness of the 'Close_log' column after applying the log transformation.
3. **Square Root Transformation Skewness**: Calculates the skewness of the 'Close_sqrt' column after applying the square root transformation.
4. **Box-Cox Transformation Skewness**: Calculates the skewness of the 'Close_boxcox' column after applying the Box-Cox transformation.

The printed results help assess how each transformation affects the distribution's symmetry and the success of skewness correction.







In [ ]:

skew_original = df_aapl['Close'].skew()
skew_log = df_aapl['Close_log'].skew()
skew_sqrt = df_aapl['Close_sqrt'].skew()
skew_boxcox = pd.Series(df_aapl['Close_boxcox']).skew()

print(f"Original Skewness: {skew_original}")
print(f"Log Transformation Skewness: {skew_log}")
print(f"Square Root Transformation Skewness: {skew_sqrt}")
print(f"Box-Cox Transformation Skewness: {skew_boxcox}")


Original Skewness: 2.5045276102319933
Log Transformation Skewness: 0.8535555176510303
Square Root Transformation Skewness: 1.6211545809555206
Box-Cox Transformation Skewness: 0.43527466713563334


In [ ]:

df_aapl['Open_log'] = np.log(df_aapl['Open'])
df_aapl['High_log'] = np.log(df_aapl['High'])
df_aapl['Low_log'] = np.log(df_aapl['Low'])
df_aapl['Adj Close_log'] = np.log(df_aapl['Adj Close'])
df_aapl['Volume_log'] = np.log(df_aapl['Volume'])


df_aapl['Open_sqrt'] = np.sqrt(df_aapl['Open'])
df_aapl['High_sqrt'] = np.sqrt(df_aapl['High'])
df_aapl['Low_sqrt'] = np.sqrt(df_aapl['Low'])
df_aapl['Adj Close_sqrt'] = np.sqrt(df_aapl['Adj Close'])
df_aapl['Volume_sqrt'] = np.sqrt(df_aapl['Volume'])

from scipy.stats import boxcox
df_aapl['Open_boxcox'], _ = boxcox(df_aapl['Open'])
df_aapl['High_boxcox'], _ = boxcox(df_aapl['High'])
df_aapl['Low_boxcox'], _ = boxcox(df_aapl['Low'])
df_aapl['Adj Close_boxcox'], _ = boxcox(df_aapl['Adj Close'])

This helps compare how the transformations reduce skewness in the data, aiming for a more normal distribution.

In [ ]:

skewness_before = df_aapl[['Open', 'High', 'Low', 'Adj Close', 'Volume']].skew()
skewness_after = df_aapl[['Open_log', 'High_log', 'Low_log', 'Adj Close_log',
                          'Open_sqrt', 'High_sqrt', 'Low_sqrt', 'Adj Close_sqrt', 'Volume_sqrt',
                          'Open_boxcox', 'High_boxcox', 'Low_boxcox', 'Adj Close_boxcox']].skew()

print("Skewness Before Transformation:\n", skewness_before)
print("\nSkewness After Transformation:\n", skewness_after)


Skewness Before Transformation:
 Open         2.504632
High         2.502208
Low          2.506714
Adj Close    2.550677
Volume       3.565699
dtype: float64

Skewness After Transformation:
 Open_log            0.482872
High_log            0.481997
Low_log             0.484246
Adj Close_log       0.494009
Open_sqrt           1.620771
High_sqrt           1.621456
Low_sqrt            1.620661
Adj Close_sqrt      1.679402
Volume_sqrt         1.299776
Open_boxcox         0.181226
High_boxcox         0.179749
Low_boxcox          0.182882
Adj Close_boxcox    0.180085
dtype: float64


- Applied Box-Cox transformation to the 'Open', 'High', 'Low', 'Adj Close', and 'Close' columns.
- Recalculated skewness after the transformation to reduce skew and normalize the data for modeling.

In [ ]:
from scipy import stats

df_aapl['Open_boxcox'], _ = stats.boxcox(df_aapl['Open'] + 1)
df_aapl['High_boxcox'], _ = stats.boxcox(df_aapl['High'] + 1)
df_aapl['Low_boxcox'], _ = stats.boxcox(df_aapl['Low'] + 1)
df_aapl['Adj Close_boxcox'], _ = stats.boxcox(df_aapl['Adj Close'] + 1)
df_aapl['Close_boxcox'], _ = stats.boxcox(df_aapl['Close'] + 1)

skewness_after_boxcox = df_aapl[['Open_boxcox', 'High_boxcox', 'Low_boxcox', 'Adj Close_boxcox', 'Close_boxcox']].skew()

print("Skewness After Box-Cox Transformation:")
print(skewness_after_boxcox)


Skewness After Box-Cox Transformation:
Open_boxcox         0.435237
High_boxcox         0.433381
Low_boxcox          0.437331
Adj Close_boxcox    0.458762
Close_boxcox        0.435275
dtype: float64


Feature Selection

In [ ]:

df_aapl_cleaned = df_aapl[['Date', 'Open', 'High', 'Low', 'Adj Close', 'Close', 'Volume',
                           'Open_boxcox', 'High_boxcox', 'Low_boxcox', 'Adj Close_boxcox',
                           'Close_boxcox']]

print(df_aapl_cleaned.head())


         Date      Open      High       Low  Adj Close     Close     Volume  \
0  1980-12-12  0.128348  0.128906  0.128348   0.098943  0.128348  469033600   
1  1980-12-15  0.122210  0.122210  0.121652   0.093781  0.121652  175884800   
2  1980-12-16  0.113281  0.113281  0.112723   0.086898  0.112723  105728000   
3  1980-12-17  0.115513  0.116071  0.115513   0.089049  0.115513   86441600   
4  1980-12-18  0.118862  0.119420  0.118862   0.091630  0.118862   73449600   

   Open_boxcox  High_boxcox  Low_boxcox  Adj Close_boxcox  Close_boxcox  
0     0.117689     0.118173    0.117674          0.092374      0.117689  
1     0.112503     0.112516    0.112016          0.087857      0.112030  
2     0.104886     0.104897    0.104395          0.081785      0.104407  
3     0.106798     0.107287    0.106786          0.083688      0.106798  
4     0.109657     0.110145    0.109644          0.085966      0.109657  


### Train Validation Test Split

The code splits the data into training, validation, and test sets. The features `X` and target `Y` are split as follows:

- 70% for training (`X_train`, `Y_train`)
- 15% for validation (`X_val`, `Y_val`)
- 15% for testing (`X_test`, `Y_test`)

The split is done using a 30% test size, followed by splitting the remaining 70% into validation and test sets without shuffling (time series data).

In [ ]:
from sklearn.model_selection import train_test_split

X = df_aapl_cleaned[['Open_boxcox', 'High_boxcox', 'Low_boxcox']]
Y = df_aapl_cleaned['Close_boxcox']

X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.3, shuffle=False)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, shuffle=False)

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}, Test set: {X_test.shape}")


Training set: (7736, 3), Validation set: (1658, 3), Test set: (1658, 3)


In [ ]:
!pip install ConfigSpace
!pip install hpbandster

## Stacked Ensemble

###Base: XGBoost, CatBoost, LGB; Final: Bagging

####initial

In [ ]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import numpy as np

# Initialize the models
xgb_model = XGBRegressor(n_estimators=186, max_depth=3, learning_rate=0.2052134234183, random_state=42)
cat_model = CatBoostRegressor(iterations=218, depth=8, learning_rate=0.1826510697721, random_state=42, verbose=0)
lgb_model = LGBMRegressor(n_estimators=171, max_depth=1, learning_rate=0.1996873779365, random_state=42)

# Train the base models
xgb_model.fit(X_train, Y_train)
cat_model.fit(X_train, Y_train)
lgb_model.fit(X_train, Y_train)

# Predict with base models
xgb_train_pred = xgb_model.predict(X_train)
cat_train_pred = cat_model.predict(X_train)
lgb_train_pred = lgb_model.predict(X_train)

xgb_val_pred = xgb_model.predict(X_val)
cat_val_pred = cat_model.predict(X_val)
lgb_val_pred = lgb_model.predict(X_val)

xgb_test_pred = xgb_model.predict(X_test)
cat_test_pred = cat_model.predict(X_test)
lgb_test_pred = lgb_model.predict(X_test)

# Stack predictions for the final Bagging model
train_stacked_preds = np.column_stack((xgb_train_pred, cat_train_pred, lgb_train_pred))
val_stacked_preds = np.column_stack((xgb_val_pred, cat_val_pred, lgb_val_pred))
test_stacked_preds = np.column_stack((xgb_test_pred, cat_test_pred, lgb_test_pred))

# Initialize the BaggingRegressor
bagging_model = BaggingRegressor(base_estimator=None, n_estimators=10, random_state=42)

# Train the Bagging model
bagging_model.fit(train_stacked_preds, Y_train)

# Bagging model predictions
bagging_train_pred = bagging_model.predict(train_stacked_preds)
bagging_val_pred = bagging_model.predict(val_stacked_preds)
bagging_test_pred = bagging_model.predict(test_stacked_preds)

# Calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Training metrics
train_metrics = calculate_metrics(Y_train, bagging_train_pred)

# Validation metrics
val_metrics = calculate_metrics(Y_val, bagging_val_pred)

# Test metrics
test_metrics = calculate_metrics(Y_test, bagging_test_pred)

# Print metrics
print("Training set metrics:")
print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000454 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

#### optuna

In [ ]:
import optuna
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import StackingRegressor, BaggingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor

# Function to optimize XGBoost model
def optimize_xgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }
    model = XGBRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize CatBoost model
def optimize_cat(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 50, 300),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'random_state': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize LightGBM model
def optimize_lgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'random_state': 42
    }
    model = LGBMRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize Extra Trees model
def optimize_et(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'random_state': 42
    }
    model = ExtraTreesRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Create Optuna study for each model and get best parameters
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(optimize_xgb, n_trials=10)

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(optimize_cat, n_trials=10)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(optimize_lgb, n_trials=10)

study_et = optuna.create_study(direction='minimize')
study_et.optimize(optimize_et, n_trials=10)

# Retrieve the best parameters from each model's optimization
best_xgb = study_xgb.best_params
best_cat = study_cat.best_params
best_lgb = study_lgb.best_params
best_et = study_et.best_params

# Define the base models using the optimized hyperparameters
base_models = [
    ('xgb', XGBRegressor(**best_xgb)),
    ('cat', CatBoostRegressor(**best_cat)),
    ('lgb', LGBMRegressor(**best_lgb))
]

# Final model (Bagging Regressor) using the optimized hyperparameters
final_model = BaggingRegressor(base_estimator=None, n_estimators=10, random_state=42)

# Create the stacking model
stack_model = StackingRegressor(estimators=base_models, final_estimator=final_model)

# Fit the model on the training data
stack_model.fit(X_train, Y_train)

# Predict and evaluate the model on training, validation, and test sets
Y_train_pred = stack_model.predict(X_train)
Y_val_pred = stack_model.predict(X_val)
Y_test_pred = stack_model.predict(X_test)

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Calculate metrics for training, validation, and test sets
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("Training set metrics:")
print(f"MAE: {train_metrics[0]:.4f}, MSE: {train_metrics[1]:.4f}, RMSE: {train_metrics[2]:.4f}, R²: {train_metrics[3]:.4f}, MAPE: {train_metrics[4]:.4f}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]:.4f}, MSE: {val_metrics[1]:.4f}, RMSE: {val_metrics[2]:.4f}, R²: {val_metrics[3]:.4f}, MAPE: {val_metrics[4]:.4f}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]:.4f}, MSE: {test_metrics[1]:.4f}, RMSE: {test_metrics[2]:.4f}, R²: {test_metrics[3]:.4f}, MAPE: {test_metrics[4]:.4f}")


[I 2025-01-06 11:25:27,714] A new study created in memory with name: no-name-2616579b-3001-4269-b87f-6f78799c1d37
[I 2025-01-06 11:25:29,602] Trial 0 finished with value: 0.25522253807755396 and parameters: {'n_estimators': 60, 'max_depth': 10, 'learning_rate': 0.044262380250005354, 'subsample': 0.6879416279893227, 'colsample_bytree': 0.8489641074833683}. Best is trial 0 with value: 0.25522253807755396.
[I 2025-01-06 11:25:29,859] Trial 1 finished with value: 0.15485925985401724 and parameters: {'n_estimators': 222, 'max_depth': 11, 'learning_rate': 0.03037353730598205, 'subsample': 0.5823745472583548, 'colsample_bytree': 0.5560291704000118}. Best is trial 1 with value: 0.15485925985401724.
[I 2025-01-06 11:25:30,191] Trial 2 finished with value: 0.12981030641195365 and parameters: {'n_estimators': 284, 'max_depth': 8, 'learning_rate': 0.09732655355931667, 'subsample': 0.7468756453023135, 'colsample_bytree': 0.799720283312214}. Best is trial 2 with value: 0.12981030641195365.
[I 2025-0

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000271 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 11:25:45,239] Trial 0 finished with value: 0.1776668591376695 and parameters: {'n_estimators': 250, 'max_depth': 5, 'learning_rate': 0.015315509881720502, 'num_leaves': 161}. Best is trial 0 with value: 0.1776668591376695.



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

[I 2025-01-06 11:25:45,408] Trial 1 finished with value: 0.17599844042645726 and parameters: {'n_estimators': 75, 'max_depth': 11, 'learning_rate': 0.05037125018308391, 'num_leaves': 69}. Best is trial 1 with value: 0.17599844042645726.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000450 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000436 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038


[I 2025-01-06 11:25:48,438] Trial 8 finished with value: 0.2759267469757147 and parameters: {'n_estimators': 147, 'max_depth': 13, 'learning_rate': 0.014996362010746295, 'num_leaves': 200}. Best is trial 4 with value: 0.1519431699819462.
[I 2025-01-06 11:25:48,556] Trial 9 finished with value: 0.18874054819669206 and parameters: {'n_estimators': 92, 'max_depth': 3, 'learning_rate': 0.043144262619154446, 'num_leaves': 106}. Best is trial 4 with value: 0.1519431699819462.
[I 2025-01-06 11:25:48,559] A new study created in memory with name: no-name-0d6c4ab8-d33b-4900-9a6e-83cd032b5dda


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 11:25:49,796] Trial 0 finished with value: 0.13535755875926947 and parameters: {'n_estimators': 148, 'max_depth': 12}. Best is trial 0 with value: 0.13535755875926947.
[I 2025-01-06 11:25:50,582] Trial 1 finished with value: 0.13601261253812885 and parameters: {'n_estimators': 141, 'max_depth': 10}. Best is trial 0 with value: 0.13535755875926947.
[I 2025-01-06 11:25:50,700] Trial 2 finished with value: 0.24425151139906612 and parameters: {'n_estimators': 85, 'max_depth': 3}. Best is trial 0 with value: 0.13535755875926947.
[I 2025-01-06 11:25:51,337] Trial 3 finished with value: 0.13550026178489788 and parameters: {'n_estimators': 80, 'max_depth': 15}. Best is trial 0 with value: 0.13535755875926947.
[I 2025-01-06 11:25:51,967] Trial 4 finished with value: 0.13736521623046208 and parameters: {'n_estimators': 209, 'max_depth': 9}. Best is trial 0 with value: 0.13535755875926947.
[I 2025-01-06 11:25:52,694] Trial 5 finished with value: 0.13538422348242324 and parameters: {

0:	learn: 0.3913647	total: 2.87ms	remaining: 826ms
1:	learn: 0.3662448	total: 6.16ms	remaining: 884ms
2:	learn: 0.3426055	total: 8.82ms	remaining: 841ms
3:	learn: 0.3207613	total: 11.8ms	remaining: 841ms
4:	learn: 0.2999313	total: 14.6ms	remaining: 828ms
5:	learn: 0.2804532	total: 17.4ms	remaining: 820ms
6:	learn: 0.2624121	total: 20.2ms	remaining: 814ms
7:	learn: 0.2454040	total: 23.1ms	remaining: 812ms
8:	learn: 0.2298180	total: 24.9ms	remaining: 775ms
9:	learn: 0.2151512	total: 27.8ms	remaining: 775ms
10:	learn: 0.2014041	total: 30.7ms	remaining: 775ms
11:	learn: 0.1883708	total: 33.5ms	remaining: 773ms
12:	learn: 0.1761885	total: 36.3ms	remaining: 771ms
13:	learn: 0.1649916	total: 39.2ms	remaining: 770ms
14:	learn: 0.1544063	total: 42.1ms	remaining: 769ms
15:	learn: 0.1446046	total: 44.9ms	remaining: 767ms
16:	learn: 0.1352519	total: 47.8ms	remaining: 765ms
17:	learn: 0.1265419	total: 50.7ms	remaining: 764ms
18:	learn: 0.1184074	total: 53.7ms	remaining: 763ms
19:	learn: 0.1108537	t

#### bohb

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import StackingRegressor, BaggingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base models (XGBoost, CatBoost, LGB)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_max_depth", 3, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("xgb_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_iterations", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_depth", 4, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("cat_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_max_depth", -1, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lgb_learning_rate", 0.01, 0.3, default_value=0.1))

    # Final model (Bagging Regressor)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("bagging_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("bagging_max_samples", 50, 300, default_value=100))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        # Base models with parameters from the configuration
        xgb = XGBRegressor(n_estimators=config["xgb_n_estimators"], max_depth=config["xgb_max_depth"], learning_rate=config["xgb_learning_rate"], random_state=42)
        cat = CatBoostRegressor(iterations=config["cat_iterations"], depth=config["cat_depth"], learning_rate=config["cat_learning_rate"], random_state=42, verbose=0)
        lgb = LGBMRegressor(n_estimators=config["lgb_n_estimators"], max_depth=config["lgb_max_depth"], learning_rate=config["lgb_learning_rate"], random_state=42)

        # Final model (Bagging Regressor)
        bagging = BaggingRegressor(base_estimator=ExtraTreesRegressor(random_state=42),
                                  n_estimators=config["bagging_n_estimators"],
                                  max_samples=config["bagging_max_samples"],
                                  random_state=42)

        # Stacked model
        stack_model = StackingRegressor(estimators=[('xgb', xgb), ('cat', cat), ('lgb', lgb)], final_estimator=bagging)

        # Fit and evaluate on validation set
        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Set up BOHB
try:
    NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.0", port=None)
    NS.start()

    worker = StackingWorker(nameserver="127.0.0.0", run_id="stacking_bohb")
    worker.run(background=True)

    bohb = BOHB(
        configspace=get_config_space(),
        run_id="stacking_bohb",
        nameserver="127.0.0.0",
        min_budget=1,
        max_budget=3
    )

    # Perform optimization
    res = bohb.run(n_iterations=10)

    # Shutdown
    bohb.shutdown()
    NS.shutdown()

    # Retrieve the best configuration
    best_config = res.get_incumbent_id()
    best_params = res.get_id2config_mapping()[best_config]["config"]

    # Build the Stacked Ensemble model with the best hyperparameters
    best_xgb = XGBRegressor(n_estimators=best_params["xgb_n_estimators"], max_depth=best_params["xgb_max_depth"], learning_rate=best_params["xgb_learning_rate"], random_state=42)
    best_cat = CatBoostRegressor(iterations=best_params["cat_iterations"], depth=best_params["cat_depth"], learning_rate=best_params["cat_learning_rate"], random_state=42, verbose=0)
    best_lgb = LGBMRegressor(n_estimators=best_params["lgb_n_estimators"], max_depth=best_params["lgb_max_depth"], learning_rate=best_params["lgb_learning_rate"], random_state=42)
    best_bagging = BaggingRegressor(base_estimator=ExtraTreesRegressor(random_state=42),
                                    n_estimators=best_params["bagging_n_estimators"],
                                    max_samples=best_params["bagging_max_samples"],
                                    random_state=42)

    best_stack_model = StackingRegressor(estimators=[('xgb', best_xgb), ('cat', best_cat), ('lgb', best_lgb)], final_estimator=best_bagging)

    # Fit the model
    best_stack_model.fit(X_train, Y_train)

    # Predict and evaluate
    Y_train_pred = best_stack_model.predict(X_train)
    Y_val_pred = best_stack_model.predict(X_val)
    Y_test_pred = best_stack_model.predict(X_test)

    # Performance metrics calculation
    train_metrics = calculate_metrics(Y_train, Y_train_pred)
    val_metrics = calculate_metrics(Y_val, Y_val_pred)
    test_metrics = calculate_metrics(Y_test, Y_test_pred)

    # Print the results
    print("Best Parameters Found by BOHB:")
    print(best_params)

    print("\nTraining set metrics:")
    print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

    print("\nValidation set metrics:")
    print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

    print("\nTest set metrics:")
    print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")

except Exception as e:
    print(f"Error occurred: {e}")
    if 'NS' in locals():
        NS.shutdown()  # Ensure NameServer is shut down on error


### Base: XGBoost, CatBoost, LGB; Final: ExtraTree

#### initial

In [ ]:
!pip install scikit-learn==1.3.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 39.2 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.0
    Uninstalling scikit-learn-1.6.0:
      Successfully uninstalled scikit-learn-1.6.0


In [ ]:
from sklearn.ensemble import StackingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# Define the base models and final model
base_models = [
    ('xgb', XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)),
    ('cat', CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)),
    ('lgb', LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42))
]

# Final model (Extra Trees)
final_model = ExtraTreesRegressor(n_estimators=100, max_depth=6, random_state=42)

# Create the stacking model
stack_model = StackingRegressor(estimators=base_models, final_estimator=final_model)

# Fit the model on the training data
stack_model.fit(X_train, Y_train)

# Predict and evaluate the model on training, validation, and test sets
Y_train_pred = stack_model.predict(X_train)
Y_val_pred = stack_model.predict(X_val)
Y_test_pred = stack_model.predict(X_test)

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Calculate metrics for training, validation, and test sets
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("Training set metrics:")
print(f"MAE: {train_metrics[0]:.4f}, MSE: {train_metrics[1]:.4f}, RMSE: {train_metrics[2]:.4f}, R²: {train_metrics[3]:.4f}, MAPE: {train_metrics[4]:.4f}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]:.4f}, MSE: {val_metrics[1]:.4f}, RMSE: {val_metrics[2]:.4f}, R²: {val_metrics[3]:.4f}, MAPE: {val_metrics[4]:.4f}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]:.4f}, MSE: {test_metrics[1]:.4f}, RMSE: {test_metrics[2]:.4f}, R²: {test_metrics[3]:.4f}, MAPE: {test_metrics[4]:.4f}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

#### optuna

In [ ]:
!pip install optuna

In [ ]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import StackingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Function to optimize XGBoost model
def optimize_xgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }
    model = XGBRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize CatBoost model
def optimize_cat(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 50, 300),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'random_state': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize LightGBM model
def optimize_lgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'random_state': 42
    }
    model = LGBMRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize Extra Trees model
def optimize_et(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'random_state': 42
    }
    model = ExtraTreesRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Create Optuna study for each model and get best parameters
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(optimize_xgb, n_trials=10)

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(optimize_cat, n_trials=10)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(optimize_lgb, n_trials=10)

study_et = optuna.create_study(direction='minimize')
study_et.optimize(optimize_et, n_trials=10)

# Retrieve the best parameters from each model's optimization
best_xgb = study_xgb.best_params
best_cat = study_cat.best_params
best_lgb = study_lgb.best_params
best_et = study_et.best_params

# Define the base models using the optimized hyperparameters
base_models = [
    ('xgb', XGBRegressor(**best_xgb)),
    ('cat', CatBoostRegressor(**best_cat)),
    ('lgb', LGBMRegressor(**best_lgb))
]

# Final model (Extra Trees) using the optimized hyperparameters
final_model = ExtraTreesRegressor(**best_et)

# Create the stacking model
stack_model = StackingRegressor(estimators=base_models, final_estimator=final_model)

# Fit the model on the training data
stack_model.fit(X_train, Y_train)

# Predict and evaluate the model on training, validation, and test sets
Y_train_pred = stack_model.predict(X_train)
Y_val_pred = stack_model.predict(X_val)
Y_test_pred = stack_model.predict(X_test)

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Calculate metrics for training, validation, and test sets
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("Training set metrics:")
print(f"MAE: {train_metrics[0]:.4f}, MSE: {train_metrics[1]:.4f}, RMSE: {train_metrics[2]:.4f}, R²: {train_metrics[3]:.4f}, MAPE: {train_metrics[4]:.4f}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]:.4f}, MSE: {val_metrics[1]:.4f}, RMSE: {val_metrics[2]:.4f}, R²: {val_metrics[3]:.4f}, MAPE: {val_metrics[4]:.4f}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]:.4f}, MSE: {test_metrics[1]:.4f}, RMSE: {test_metrics[2]:.4f}, R²: {test_metrics[3]:.4f}, MAPE: {test_metrics[4]:.4f}")


[I 2025-01-06 10:37:38,438] A new study created in memory with name: no-name-af618b6c-cab9-4051-b194-5236915f0653
[I 2025-01-06 10:37:49,736] Trial 0 finished with value: 0.13362690057554638 and parameters: {'n_estimators': 153, 'max_depth': 13, 'learning_rate': 0.08349659034341878, 'subsample': 0.7089163362210642, 'colsample_bytree': 0.7115539892049886}. Best is trial 0 with value: 0.13362690057554638.
[I 2025-01-06 10:38:06,073] Trial 1 finished with value: 0.13500525882443987 and parameters: {'n_estimators': 272, 'max_depth': 14, 'learning_rate': 0.04130889852990376, 'subsample': 0.6134598526524628, 'colsample_bytree': 0.8463735838187902}. Best is trial 0 with value: 0.13362690057554638.
[I 2025-01-06 10:38:09,866] Trial 2 finished with value: 0.1972025423881619 and parameters: {'n_estimators': 111, 'max_depth': 7, 'learning_rate': 0.03219001273161097, 'subsample': 0.9552962617020786, 'colsample_bytree': 0.6619517003642758}. Best is trial 0 with value: 0.13362690057554638.
[I 2025-0

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038


[I 2025-01-06 10:38:42,639] Trial 0 finished with value: 0.16476701917785025 and parameters: {'n_estimators': 274, 'max_depth': 13, 'learning_rate': 0.016239561865869383, 'num_leaves': 137}. Best is trial 0 with value: 0.16476701917785025.
[I 2025-01-06 10:38:42,803] Trial 1 finished with value: 0.44321442724127497 and parameters: {'n_estimators': 83, 'max_depth': 9, 'learning_rate': 0.016327536057336545, 'num_leaves': 120}. Best is trial 0 with value: 0.16476701917785025.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000319 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000281 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038


[I 2025-01-06 10:38:42,938] Trial 2 finished with value: 0.2367712815854498 and parameters: {'n_estimators': 149, 'max_depth': 9, 'learning_rate': 0.017516008305941383, 'num_leaves': 38}. Best is trial 0 with value: 0.16476701917785025.
[I 2025-01-06 10:38:43,124] Trial 3 finished with value: 0.1532441231596569 and parameters: {'n_estimators': 223, 'max_depth': 9, 'learning_rate': 0.03141565833100569, 'num_leaves': 37}. Best is trial 3 with value: 0.1532441231596569.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 10:38:45,127] Trial 8 finished with value: 0.15232038928039732 and parameters: {'n_estimators': 204, 'max_depth': 4, 'learning_rate': 0.04553898459886395, 'num_leaves': 111}. Best is trial 4 with value: 0.15187782336023237.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000489 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 10:38:45,394] Trial 9 finished with value: 0.15210306261875026 and parameters: {'n_estimators': 210, 'max_depth': 4, 'learning_rate': 0.09942636713335379, 'num_leaves': 179}. Best is trial 4 with value: 0.15187782336023237.
[I 2025-01-06 10:38:45,397] A new study created in memory with name: no-name-8d246dcc-4778-453c-941e-e0f98c4926bb
[I 2025-01-06 10:38:48,229] Trial 0 finished with value: 0.13531632549288777 and parameters: {'n_estimators': 292, 'max_depth': 13}. Best is trial 0 with value: 0.13531632549288777.
[I 2025-01-06 10:38:48,676] Trial 1 finished with value: 0.16779211430277166 and parameters: {'n_estimators': 141, 'max_depth': 5}. Best is trial 0 with value: 0.13531632549288777.
[I 2025-01-06 10:38:49,686] Trial 2 finished with value: 0.1354063749711454 and parameters: {'n_estimators': 108, 'max_depth': 15}. Best is trial 0 with value: 0.13531632549288777.
[I 2025-01-06 10:38:50,045] Trial 3 finished with value: 0.19698697068227167 and parameters: {'n_estimat

0:	learn: 0.3793047	total: 11.9ms	remaining: 2.37s
1:	learn: 0.3442393	total: 24.3ms	remaining: 2.4s
2:	learn: 0.3120603	total: 35.2ms	remaining: 2.31s
3:	learn: 0.2831779	total: 46.1ms	remaining: 2.26s
4:	learn: 0.2571239	total: 62.3ms	remaining: 2.43s
5:	learn: 0.2333375	total: 73.4ms	remaining: 2.37s
6:	learn: 0.2115394	total: 86.8ms	remaining: 2.39s
7:	learn: 0.1919733	total: 97.7ms	remaining: 2.35s
8:	learn: 0.1743889	total: 109ms	remaining: 2.31s
9:	learn: 0.1581988	total: 120ms	remaining: 2.28s
10:	learn: 0.1436153	total: 132ms	remaining: 2.27s
11:	learn: 0.1304176	total: 143ms	remaining: 2.24s
12:	learn: 0.1184083	total: 154ms	remaining: 2.22s
13:	learn: 0.1075124	total: 166ms	remaining: 2.2s
14:	learn: 0.0976525	total: 177ms	remaining: 2.18s
15:	learn: 0.0886281	total: 187ms	remaining: 2.15s
16:	learn: 0.0804705	total: 199ms	remaining: 2.14s
17:	learn: 0.0731055	total: 211ms	remaining: 2.13s
18:	learn: 0.0664076	total: 225ms	remaining: 2.14s
19:	learn: 0.0604096	total: 235ms	r

In [ ]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import StackingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Function to optimize XGBoost model
def optimize_xgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }
    model = XGBRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize CatBoost model
def optimize_cat(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 50, 300),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'random_state': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize LightGBM model
def optimize_lgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'random_state': 42
    }
    model = LGBMRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize Extra Trees model
def optimize_et(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'random_state': 42
    }
    model = ExtraTreesRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Create Optuna study for each model and get best parameters
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(optimize_xgb, n_trials=30)

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(optimize_cat, n_trials=30)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(optimize_lgb, n_trials=30)

study_et = optuna.create_study(direction='minimize')
study_et.optimize(optimize_et, n_trials=30)

# Retrieve the best parameters from each model's optimization
best_xgb = study_xgb.best_params
best_cat = study_cat.best_params
best_lgb = study_lgb.best_params
best_et = study_et.best_params

# Define the base models using the optimized hyperparameters
base_models = [
    ('xgb', XGBRegressor(**best_xgb)),
    ('cat', CatBoostRegressor(**best_cat)),
    ('lgb', LGBMRegressor(**best_lgb))
]

# Final model (Extra Trees) using the optimized hyperparameters
final_model = ExtraTreesRegressor(**best_et)

# Create the stacking model
stack_model = StackingRegressor(estimators=base_models, final_estimator=final_model)

# Fit the model on the training data
stack_model.fit(X_train, Y_train)

# Predict and evaluate the model on training, validation, and test sets
Y_train_pred = stack_model.predict(X_train)
Y_val_pred = stack_model.predict(X_val)
Y_test_pred = stack_model.predict(X_test)

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Calculate metrics for training, validation, and test sets
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("Training set metrics:")
print(f"MAE: {train_metrics[0]:.4f}, MSE: {train_metrics[1]:.4f}, RMSE: {train_metrics[2]:.4f}, R²: {train_metrics[3]:.4f}, MAPE: {train_metrics[4]:.4f}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]:.4f}, MSE: {val_metrics[1]:.4f}, RMSE: {val_metrics[2]:.4f}, R²: {val_metrics[3]:.4f}, MAPE: {val_metrics[4]:.4f}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]:.4f}, MSE: {test_metrics[1]:.4f}, RMSE: {test_metrics[2]:.4f}, R²: {test_metrics[3]:.4f}, MAPE: {test_metrics[4]:.4f}")


[I 2025-01-06 10:41:16,928] A new study created in memory with name: no-name-f5b84011-c24d-4dbb-8ffa-c28609a3af8f
[I 2025-01-06 10:41:20,982] Trial 0 finished with value: 0.13334555389669095 and parameters: {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.046717863188397636, 'subsample': 0.8061706513405075, 'colsample_bytree': 0.7434524624598643}. Best is trial 0 with value: 0.13334555389669095.
[I 2025-01-06 10:41:21,325] Trial 1 finished with value: 0.14531793475364885 and parameters: {'n_estimators': 236, 'max_depth': 14, 'learning_rate': 0.03685413135315542, 'subsample': 0.5691132183224283, 'colsample_bytree': 0.5980277823924793}. Best is trial 0 with value: 0.13334555389669095.
[I 2025-01-06 10:41:21,452] Trial 2 finished with value: 0.1636986963901225 and parameters: {'n_estimators': 118, 'max_depth': 14, 'learning_rate': 0.04468266183461569, 'subsample': 0.9950030685028461, 'colsample_bytree': 0.6587818561218375}. Best is trial 0 with value: 0.13334555389669095.
[I 2025-

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000461 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 10:42:14,819] Trial 0 finished with value: 0.15236048741178773 and parameters: {'n_estimators': 241, 'max_depth': 4, 'learning_rate': 0.040741869150245, 'num_leaves': 85}. Best is trial 0 with value: 0.15236048741178773.



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

[I 2025-01-06 10:42:15,571] Trial 1 finished with value: 0.15201721390616352 and parameters: {'n_estimators': 271, 'max_depth': 15, 'learning_rate': 0.03711049208864039, 'num_leaves': 90}. Best is trial 1 with value: 0.15201721390616352.
[I 2025-01-06 10:42:15,776] Trial 2 finished with value: 0.17527369514901073 and parameters: {'n_estimators': 149, 'max_depth': 4, 'learning_rate': 0.027340799691632344, 'num_leaves': 43}. Best is trial 1 with value: 0.15201721390616352.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000272 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 10:42:25,385] Trial 28 finished with value: 0.151910383414953 and parameters: {'n_estimators': 216, 'max_depth': 9, 'learning_rate': 0.08528014324993391, 'num_leaves': 162}. Best is trial 28 with value: 0.151910383414953.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-01-06 10:42:25,670] Trial 29 finished with value: 0.1520108658952392 and parameters: {'n_estimators': 211, 'max_depth': 5, 'learning_rate': 0.0842349561645166, 'num_leaves': 127}. Best is trial 28 with value: 0.151910383414953.
[I 2025-01-06 10:42:25,673] A new study created in memory with name: no-name-19531585-d46e-48e0-b3cc-8d9b5619ecf7
[I 2025-01-06 10:42:25,862] Trial 0 finished with value: 0.13782713013905107 and parameters: {'n_estimators': 60, 'max_depth': 9}. Best is trial 0 with value: 0.13782713013905107.
[I 2025-01-06 10:42:26,193] Trial 1 finished with value: 0.19718930040856877 and parameters: {'n_estimators': 217, 'max_depth': 4}. Best is trial 0 with value: 0.13782713013905107.
[I 2025-01-06 10:42:26,496] Trial 2 finished with value: 0.16842966003980508 and parameters: {'n_estimators': 165, 'max_depth': 5}. Best is trial 0 with value: 0.13782713013905107.
[I 2025-01-06 10:42:27,384] Trial 3 finished with value: 0.13537583998146852 and parameters: {'n_estimators'

0:	learn: 0.3920773	total: 11.7ms	remaining: 3.26s
1:	learn: 0.3676844	total: 22.9ms	remaining: 3.18s
2:	learn: 0.3445452	total: 34.4ms	remaining: 3.17s
3:	learn: 0.3229349	total: 45.4ms	remaining: 3.13s
4:	learn: 0.3029377	total: 56.4ms	remaining: 3.1s
5:	learn: 0.2841531	total: 67.1ms	remaining: 3.06s
6:	learn: 0.2662737	total: 77.9ms	remaining: 3.04s
7:	learn: 0.2496884	total: 88.5ms	remaining: 3.01s
8:	learn: 0.2342541	total: 99.3ms	remaining: 2.99s
9:	learn: 0.2195968	total: 112ms	remaining: 3.02s
10:	learn: 0.2058271	total: 123ms	remaining: 3.02s
11:	learn: 0.1929379	total: 134ms	remaining: 3s
12:	learn: 0.1809908	total: 138ms	remaining: 2.84s
13:	learn: 0.1697606	total: 149ms	remaining: 2.82s
14:	learn: 0.1592009	total: 160ms	remaining: 2.82s
15:	learn: 0.1492577	total: 171ms	remaining: 2.82s
16:	learn: 0.1400753	total: 182ms	remaining: 2.82s
17:	learn: 0.1313935	total: 194ms	remaining: 2.82s
18:	learn: 0.1233531	total: 205ms	remaining: 2.81s
19:	learn: 0.1157208	total: 222ms	re

In [ ]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import StackingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Function to optimize XGBoost model
def optimize_xgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }
    model = XGBRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize CatBoost model
def optimize_cat(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 50, 300),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'random_state': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize LightGBM model
def optimize_lgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'random_state': 42
    }
    model = LGBMRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Function to optimize Extra Trees model
def optimize_et(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'random_state': 42
    }
    model = ExtraTreesRegressor(**param)
    model.fit(X_train, Y_train)
    Y_val_pred = model.predict(X_val)
    return mean_absolute_error(Y_val, Y_val_pred)

# Create Optuna study for each model and get best parameters
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(optimize_xgb, n_trials=50)

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(optimize_cat, n_trials=50)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(optimize_lgb, n_trials=50)

study_et = optuna.create_study(direction='minimize')
study_et.optimize(optimize_et, n_trials=50)

# Retrieve the best parameters from each model's optimization
best_xgb = study_xgb.best_params
best_cat = study_cat.best_params
best_lgb = study_lgb.best_params
best_et = study_et.best_params

# Define the base models using the optimized hyperparameters
base_models = [
    ('xgb', XGBRegressor(**best_xgb)),
    ('cat', CatBoostRegressor(**best_cat)),
    ('lgb', LGBMRegressor(**best_lgb))
]

# Final model (Extra Trees) using the optimized hyperparameters
final_model = ExtraTreesRegressor(**best_et)

# Create the stacking model
stack_model = StackingRegressor(estimators=base_models, final_estimator=final_model)

# Fit the model on the training data
stack_model.fit(X_train, Y_train)

# Predict and evaluate the model on training, validation, and test sets
Y_train_pred = stack_model.predict(X_train)
Y_val_pred = stack_model.predict(X_val)
Y_test_pred = stack_model.predict(X_test)

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Calculate metrics for training, validation, and test sets
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("Training set metrics:")
print(f"MAE: {train_metrics[0]:.4f}, MSE: {train_metrics[1]:.4f}, RMSE: {train_metrics[2]:.4f}, R²: {train_metrics[3]:.4f}, MAPE: {train_metrics[4]:.4f}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]:.4f}, MSE: {val_metrics[1]:.4f}, RMSE: {val_metrics[2]:.4f}, R²: {val_metrics[3]:.4f}, MAPE: {val_metrics[4]:.4f}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]:.4f}, MSE: {test_metrics[1]:.4f}, RMSE: {test_metrics[2]:.4f}, R²: {test_metrics[3]:.4f}, MAPE: {test_metrics[4]:.4f}")


[I 2025-01-06 10:45:24,750] A new study created in memory with name: no-name-0c2ef1ac-f3f3-4ce5-89f9-69575736b86c
[I 2025-01-06 10:45:26,525] Trial 0 finished with value: 0.26294253166886256 and parameters: {'n_estimators': 131, 'max_depth': 9, 'learning_rate': 0.01990086802095713, 'subsample': 0.7088757553793625, 'colsample_bytree': 0.6183311815147846}. Best is trial 0 with value: 0.26294253166886256.
[I 2025-01-06 10:45:28,584] Trial 1 finished with value: 0.15890061188453802 and parameters: {'n_estimators': 55, 'max_depth': 7, 'learning_rate': 0.09867236940394383, 'subsample': 0.9619945466058075, 'colsample_bytree': 0.8455330052660328}. Best is trial 1 with value: 0.15890061188453802.
[I 2025-01-06 10:45:28,904] Trial 2 finished with value: 0.1493556836725388 and parameters: {'n_estimators': 291, 'max_depth': 10, 'learning_rate': 0.025687326438949262, 'subsample': 0.6017757680303668, 'colsample_bytree': 0.5430731446233479}. Best is trial 2 with value: 0.1493556836725388.
[I 2025-01-

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000482 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038


[I 2025-01-06 10:47:27,553] Trial 0 finished with value: 0.2910785270807799 and parameters: {'n_estimators': 132, 'max_depth': 13, 'learning_rate': 0.015871476137498633, 'num_leaves': 69}. Best is trial 0 with value: 0.2910785270807799.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000469 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 10:47:27,928] Trial 1 finished with value: 0.15872841852197492 and parameters: {'n_estimators': 177, 'max_depth': 6, 'learning_rate': 0.0285087649363832, 'num_leaves': 93}. Best is trial 1 with value: 0.15872841852197492.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-01-06 10:47:54,993] Trial 49 finished with value: 0.15195380210154544 and parameters: {'n_estimators': 291, 'max_depth': 6, 'learning_rate': 0.08742936066750037, 'num_leaves': 177}. Best is trial 44 with value: 0.15187778959488463.
[I 2025-01-06 10:47:54,995] A new study created in memory with name: no-name-7eb50dc7-d620-460c-b455-4dba20d9a396


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-01-06 10:47:55,130] Trial 0 finished with value: 0.20270861646355534 and parameters: {'n_estimators': 74, 'max_depth': 4}. Best is trial 0 with value: 0.20270861646355534.
[I 2025-01-06 10:47:55,514] Trial 1 finished with value: 0.2430468833364773 and parameters: {'n_estimators': 280, 'max_depth': 3}. Best is trial 0 with value: 0.20270861646355534.
[I 2025-01-06 10:47:55,716] Trial 2 finished with value: 0.13986051235481575 and parameters: {'n_estimators': 72, 'max_depth': 8}. Best is trial 2 with value: 0.13986051235481575.
[I 2025-01-06 10:47:56,202] Trial 3 finished with value: 0.13537844271740954 and parameters: {'n_estimators': 60, 'max_depth': 15}. Best is trial 3 with value: 0.13537844271740954.
[I 2025-01-06 10:47:57,633] Trial 4 finished with value: 0.13533368978637003 and parameters: {'n_estimators': 183, 'max_depth': 15}. Best is trial 4 with value: 0.13533368978637003.
[I 2025-01-06 10:47:57,769] Trial 5 finished with value: 0.1581160927656886 and parameters: {'n_e

0:	learn: 0.3802144	total: 12ms	remaining: 3.55s
1:	learn: 0.3458835	total: 23.5ms	remaining: 3.48s
2:	learn: 0.3143029	total: 35.6ms	remaining: 3.5s
3:	learn: 0.2858893	total: 47.6ms	remaining: 3.5s
4:	learn: 0.2601972	total: 59ms	remaining: 3.46s
5:	learn: 0.2366841	total: 69.8ms	remaining: 3.39s
6:	learn: 0.2150867	total: 81.2ms	remaining: 3.37s
7:	learn: 0.1956504	total: 93ms	remaining: 3.37s
8:	learn: 0.1781407	total: 105ms	remaining: 3.36s
9:	learn: 0.1619856	total: 116ms	remaining: 3.35s
10:	learn: 0.1473124	total: 128ms	remaining: 3.34s
11:	learn: 0.1340733	total: 139ms	remaining: 3.31s
12:	learn: 0.1220101	total: 150ms	remaining: 3.3s
13:	learn: 0.1110290	total: 161ms	remaining: 3.27s
14:	learn: 0.1010776	total: 173ms	remaining: 3.26s
15:	learn: 0.0919464	total: 184ms	remaining: 3.25s
16:	learn: 0.0836799	total: 196ms	remaining: 3.24s
17:	learn: 0.0761799	total: 207ms	remaining: 3.22s
18:	learn: 0.0694300	total: 224ms	remaining: 3.29s
19:	learn: 0.0633162	total: 236ms	remainin

#### bohb

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base models (XGBoost, CatBoost, LGB)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_max_depth", 3, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("xgb_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_iterations", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_depth", 4, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("cat_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_max_depth", -1, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lgb_learning_rate", 0.01, 0.3, default_value=0.1))

    # Final model (Extra Trees)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("et_max_depth", 3, 15, default_value=6))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        # Base models with parameters from the configuration
        xgb = XGBRegressor(n_estimators=config["xgb_n_estimators"], max_depth=config["xgb_max_depth"], learning_rate=config["xgb_learning_rate"], random_state=42)
        cat = CatBoostRegressor(iterations=config["cat_iterations"], depth=config["cat_depth"], learning_rate=config["cat_learning_rate"], random_state=42, verbose=0)
        lgb = LGBMRegressor(n_estimators=config["lgb_n_estimators"], max_depth=config["lgb_max_depth"], learning_rate=config["lgb_learning_rate"], random_state=42)

        # Final model (Extra Trees)
        et = ExtraTreesRegressor(max_depth=config["et_max_depth"], random_state=42)

        # Stacked model
        stack_model = StackingRegressor(estimators=[('xgb', xgb), ('cat', cat), ('lgb', lgb)], final_estimator=et)

        # Fit and evaluate on validation set
        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Set up BOHB
try:
    NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.0", port=None)
    NS.start()

    worker = StackingWorker(nameserver="127.0.0.0", run_id="stacking_bohb")
    worker.run(background=True)

    bohb = BOHB(
        configspace=get_config_space(),
        run_id="stacking_bohb",
        nameserver="127.0.0.0",
        min_budget=1,
        max_budget=3
    )

    # Perform optimization
    res = bohb.run(n_iterations=10)

    # Shutdown
    bohb.shutdown()
    NS.shutdown()

    # Retrieve the best configuration
    best_config = res.get_incumbent_id()
    best_params = res.get_id2config_mapping()[best_config]["config"]

    # Build the Stacked Ensemble model with the best hyperparameters
    best_xgb = XGBRegressor(n_estimators=best_params["xgb_n_estimators"], max_depth=best_params["xgb_max_depth"], learning_rate=best_params["xgb_learning_rate"], random_state=42)
    best_cat = CatBoostRegressor(iterations=best_params["cat_iterations"], depth=best_params["cat_depth"], learning_rate=best_params["cat_learning_rate"], random_state=42, verbose=0)
    best_lgb = LGBMRegressor(n_estimators=best_params["lgb_n_estimators"], max_depth=best_params["lgb_max_depth"], learning_rate=best_params["lgb_learning_rate"], random_state=42)
    best_et = ExtraTreesRegressor(max_depth=best_params["et_max_depth"], random_state=42)

    best_stack_model = StackingRegressor(estimators=[('xgb', best_xgb), ('cat', best_cat), ('lgb', best_lgb)], final_estimator=best_et)

    # Fit the model
    best_stack_model.fit(X_train, Y_train)

    # Predict and evaluate
    Y_train_pred = best_stack_model.predict(X_train)
    Y_val_pred = best_stack_model.predict(X_val)
    Y_test_pred = best_stack_model.predict(X_test)

    # Performance metrics calculation
    train_metrics = calculate_metrics(Y_train, Y_train_pred)
    val_metrics = calculate_metrics(Y_val, Y_val_pred)
    test_metrics = calculate_metrics(Y_test, Y_test_pred)

    # Print the results
    print("Best Parameters Found by BOHB:")
    print(best_params)

    print("\nTraining set metrics:")
    print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

    print("\nValidation set metrics:")
    print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

    print("\nTest set metrics:")
    print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")

except Exception as e:
    print(f"Error occurred: {e}")
    if 'NS' in locals():
        NS.shutdown()  # Ensure NameServer is shut down on error


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000437 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base models (XGBoost, CatBoost, LGB)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_max_depth", 3, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("xgb_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_iterations", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_depth", 4, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("cat_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_max_depth", -1, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lgb_learning_rate", 0.01, 0.3, default_value=0.1))

    # Final model (Extra Trees)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("et_max_depth", 3, 15, default_value=6))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        # Base models with parameters from the configuration
        xgb = XGBRegressor(n_estimators=config["xgb_n_estimators"], max_depth=config["xgb_max_depth"], learning_rate=config["xgb_learning_rate"], random_state=42)
        cat = CatBoostRegressor(iterations=config["cat_iterations"], depth=config["cat_depth"], learning_rate=config["cat_learning_rate"], random_state=42, verbose=0)
        lgb = LGBMRegressor(n_estimators=config["lgb_n_estimators"], max_depth=config["lgb_max_depth"], learning_rate=config["lgb_learning_rate"], random_state=42)

        # Final model (Extra Trees)
        et = ExtraTreesRegressor(max_depth=config["et_max_depth"], random_state=42)

        # Stacked model
        stack_model = StackingRegressor(estimators=[('xgb', xgb), ('cat', cat), ('lgb', lgb)], final_estimator=et)

        # Fit and evaluate on validation set
        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Set up BOHB
try:
    NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.0", port=None)
    NS.start()

    worker = StackingWorker(nameserver="127.0.0.0", run_id="stacking_bohb")
    worker.run(background=True)

    bohb = BOHB(
        configspace=get_config_space(),
        run_id="stacking_bohb",
        nameserver="127.0.0.0",
        min_budget=1,
        max_budget=3
    )

    # Perform optimization
    res = bohb.run(n_iterations=25)

    # Shutdown
    bohb.shutdown()
    NS.shutdown()

    # Retrieve the best configuration
    best_config = res.get_incumbent_id()
    best_params = res.get_id2config_mapping()[best_config]["config"]

    # Build the Stacked Ensemble model with the best hyperparameters
    best_xgb = XGBRegressor(n_estimators=best_params["xgb_n_estimators"], max_depth=best_params["xgb_max_depth"], learning_rate=best_params["xgb_learning_rate"], random_state=42)
    best_cat = CatBoostRegressor(iterations=best_params["cat_iterations"], depth=best_params["cat_depth"], learning_rate=best_params["cat_learning_rate"], random_state=42, verbose=0)
    best_lgb = LGBMRegressor(n_estimators=best_params["lgb_n_estimators"], max_depth=best_params["lgb_max_depth"], learning_rate=best_params["lgb_learning_rate"], random_state=42)
    best_et = ExtraTreesRegressor(max_depth=best_params["et_max_depth"], random_state=42)

    best_stack_model = StackingRegressor(estimators=[('xgb', best_xgb), ('cat', best_cat), ('lgb', best_lgb)], final_estimator=best_et)

    # Fit the model
    best_stack_model.fit(X_train, Y_train)

    # Predict and evaluate
    Y_train_pred = best_stack_model.predict(X_train)
    Y_val_pred = best_stack_model.predict(X_val)
    Y_test_pred = best_stack_model.predict(X_test)

    # Performance metrics calculation
    train_metrics = calculate_metrics(Y_train, Y_train_pred)
    val_metrics = calculate_metrics(Y_val, Y_val_pred)
    test_metrics = calculate_metrics(Y_test, Y_test_pred)

    # Print the results
    print("Best Parameters Found by BOHB:")
    print(best_params)

    print("\nTraining set metrics:")
    print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

    print("\nValidation set metrics:")
    print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

    print("\nTest set metrics:")
    print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")

except Exception as e:
    print(f"Error occurred: {e}")
    if 'NS' in locals():
        NS.shutdown()  # Ensure NameServer is shut down on error


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000442 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

### Base: XGBoost, CatBoost, LGB; Final: DecisionTree

#### initial

In [ ]:
# Import necessary libraries
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Initialize base models
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
catboost_model = CatBoostRegressor(learning_rate=0.1, iterations=1000, depth=6, random_state=42, verbose=0)
lgb_model = lgb.LGBMRegressor(objective='regression', random_state=42)

# Train base models
xgb_model.fit(X_train, Y_train)
catboost_model.fit(X_train, Y_train)
lgb_model.fit(X_train, Y_train)

# Make predictions on training set
xgb_train_pred = xgb_model.predict(X_train)
catboost_train_pred = catboost_model.predict(X_train)
lgb_train_pred = lgb_model.predict(X_train)

# Combine predictions from base models (stacking) for training set
train_predictions = np.column_stack((xgb_train_pred, catboost_train_pred, lgb_train_pred))

# Train the final model (DecisionTree) on the stacked predictions
final_model = DecisionTreeRegressor(random_state=42)
final_model.fit(train_predictions, Y_train)

# Evaluate training set metrics
train_final_pred = final_model.predict(train_predictions)
train_mae = mean_absolute_error(Y_train, train_final_pred)
train_mse = mean_squared_error(Y_train, train_final_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(Y_train, train_final_pred)
train_mape = mean_absolute_percentage_error(Y_train, train_final_pred)

# Print training set metrics
print("Training set metrics:")
print(f"MAE: {train_mae:.4f}, MSE: {train_mse:.4f}, RMSE: {train_rmse:.4f}, R²: {train_r2:.4f}, MAPE: {train_mape:.4f}")

# Make predictions on validation set
xgb_val_pred = xgb_model.predict(X_val)
catboost_val_pred = catboost_model.predict(X_val)
lgb_val_pred = lgb_model.predict(X_val)

# Combine predictions from base models (stacking) for validation set
val_predictions = np.column_stack((xgb_val_pred, catboost_val_pred, lgb_val_pred))

# Evaluate validation set metrics
val_final_pred = final_model.predict(val_predictions)
val_mae = mean_absolute_error(Y_val, val_final_pred)
val_mse = mean_squared_error(Y_val, val_final_pred)
val_rmse = np.sqrt(val_mse)
val_r2 = r2_score(Y_val, val_final_pred)
val_mape = mean_absolute_percentage_error(Y_val, val_final_pred)

# Print validation set metrics
print("Validation set metrics:")
print(f"MAE: {val_mae:.4f}, MSE: {val_mse:.4f}, RMSE: {val_rmse:.4f}, R²: {val_r2:.4f}, MAPE: {val_mape:.4f}")

# Make predictions on test set
xgb_test_pred = xgb_model.predict(X_test)
catboost_test_pred = catboost_model.predict(X_test)
lgb_test_pred = lgb_model.predict(X_test)

# Combine predictions from base models (stacking) for test set
test_predictions = np.column_stack((xgb_test_pred, catboost_test_pred, lgb_test_pred))

# Evaluate test set metrics
test_final_pred = final_model.predict(test_predictions)
test_mae = mean_absolute_error(Y_test, test_final_pred)
test_mse = mean_squared_error(Y_test, test_final_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(Y_test, test_final_pred)
test_mape = mean_absolute_percentage_error(Y_test, test_final_pred)

# Print test set metrics
print("Test set metrics:")
print(f"MAE: {test_mae:.4f}, MSE: {test_mse:.4f}, RMSE: {test_rmse:.4f}, R²: {test_r2:.4f}, MAPE: {test_mape:.4f}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000299 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
Training set metrics:
MAE: 0.0008, MSE: 0.0000, RMSE: 0.0020, R²: 1.0000, MAPE: 0.0019
Validation set metrics:
MAE: 0.1354, MSE: 0.0239, RMSE: 0.1546, R²: -3.0657, MAPE: 0.0758
Test set metrics:
MAE: 0.4034, MSE: 0.1687, RMSE: 0.4108, R²: -26.9577, MAPE: 0.1989


#### Optuna

In [ ]:
import optuna
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Objective function for XGBoost model optimization
def objective_xgb(trial):
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'alpha': trial.suggest_float('alpha', 0.01, 1.0),
        'lambda': trial.suggest_float('lambda', 0.01, 1.0),
    }

    model = xgb.XGBRegressor(**params)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(Y_val, y_pred))
    return rmse

# Objective function for CatBoost model optimization
def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': 0
    }

    model = CatBoostRegressor(**params)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(Y_val, y_pred))
    return rmse

# Objective function for LightGBM model optimization
def objective_lgb(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(Y_val, y_pred))
    return rmse

# Optuna study for each model
xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(objective_xgb, n_trials=50)

catboost_study = optuna.create_study(direction='minimize')
catboost_study.optimize(objective_catboost, n_trials=50)

lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(objective_lgb, n_trials=50)

# Extract the best hyperparameters from Optuna
best_xgb_params = xgb_study.best_trial.params
best_catboost_params = catboost_study.best_trial.params
best_lgb_params = lgb_study.best_trial.params

# Train the base models with the best hyperparameters found by Optuna
xgb_model = xgb.XGBRegressor(**best_xgb_params)
catboost_model = CatBoostRegressor(**best_catboost_params)
lgb_model = lgb.LGBMRegressor(**best_lgb_params)

xgb_model.fit(X_train, Y_train)
catboost_model.fit(X_train, Y_train)
lgb_model.fit(X_train, Y_train)

# Make predictions on training set
xgb_train_pred = xgb_model.predict(X_train)
catboost_train_pred = catboost_model.predict(X_train)
lgb_train_pred = lgb_model.predict(X_train)

# Combine predictions from base models (stacking) for training set
train_predictions = np.column_stack((xgb_train_pred, catboost_train_pred, lgb_train_pred))

# Train the final model (DecisionTree) on the stacked predictions
final_model = DecisionTreeRegressor(random_state=42)
final_model.fit(train_predictions, Y_train)

# Evaluate training set metrics
train_final_pred = final_model.predict(train_predictions)
train_mae = mean_absolute_error(Y_train, train_final_pred)
train_mse = mean_squared_error(Y_train, train_final_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(Y_train, train_final_pred)
train_mape = mean_absolute_percentage_error(Y_train, train_final_pred)

# Print training set metrics
print("Training set metrics:")
print(f"MAE: {train_mae:.4f}, MSE: {train_mse:.4f}, RMSE: {train_rmse:.4f}, R²: {train_r2:.4f}, MAPE: {train_mape:.4f}")

# Make predictions on validation set
xgb_val_pred = xgb_model.predict(X_val)
catboost_val_pred = catboost_model.predict(X_val)
lgb_val_pred = lgb_model.predict(X_val)

# Combine predictions from base models (stacking) for validation set
val_predictions = np.column_stack((xgb_val_pred, catboost_val_pred, lgb_val_pred))

# Evaluate validation set metrics
val_final_pred = final_model.predict(val_predictions)
val_mae = mean_absolute_error(Y_val, val_final_pred)
val_mse = mean_squared_error(Y_val, val_final_pred)
val_rmse = np.sqrt(val_mse)
val_r2 = r2_score(Y_val, val_final_pred)
val_mape = mean_absolute_percentage_error(Y_val, val_final_pred)

# Print validation set metrics
print("Validation set metrics:")
print(f"MAE: {val_mae:.4f}, MSE: {val_mse:.4f}, RMSE: {val_rmse:.4f}, R²: {val_r2:.4f}, MAPE: {val_mape:.4f}")

# Make predictions on test set
xgb_test_pred = xgb_model.predict(X_test)
catboost_test_pred = catboost_model.predict(X_test)
lgb_test_pred = lgb_model.predict(X_test)

# Combine predictions from base models (stacking) for test set
test_predictions = np.column_stack((xgb_test_pred, catboost_test_pred, lgb_test_pred))

# Evaluate test set metrics
test_final_pred = final_model.predict(test_predictions)
test_mae = mean_absolute_error(Y_test, test_final_pred)
test_mse = mean_squared_error(Y_test, test_final_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(Y_test, test_final_pred)
test_mape = mean_absolute_percentage_error(Y_test, test_final_pred)

# Print test set metrics
print("Test set metrics:")
print(f"MAE: {test_mae:.4f}, MSE: {test_mse:.4f}, RMSE: {test_rmse:.4f}, R²: {test_r2:.4f}, MAPE: {test_mape:.4f}")


[I 2025-01-06 08:57:16,766] A new study created in memory with name: no-name-5361c879-969e-418e-9123-585c25ae088d
[I 2025-01-06 08:57:16,855] Trial 0 finished with value: 0.19719814748208914 and parameters: {'learning_rate': 0.0865860572697126, 'max_depth': 10, 'n_estimators': 109, 'subsample': 0.6341228044300329, 'colsample_bytree': 0.8905740563954476, 'alpha': 0.5964436649601492, 'lambda': 0.6182781316494494}. Best is trial 0 with value: 0.19719814748208914.
[I 2025-01-06 08:57:16,981] Trial 1 finished with value: 0.1947247308710545 and parameters: {'learning_rate': 0.23982811614746047, 'max_depth': 9, 'n_estimators': 259, 'subsample': 0.8713045620765684, 'colsample_bytree': 0.919108935023545, 'alpha': 0.7768833437913576, 'lambda': 0.6711986469885779}. Best is trial 1 with value: 0.1947247308710545.
[I 2025-01-06 08:57:17,093] Trial 2 finished with value: 0.19291797953064715 and parameters: {'learning_rate': 0.10087719710488752, 'max_depth': 6, 'n_estimators': 279, 'subsample': 0.981

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000340 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 09:00:10,487] Trial 0 finished with value: 0.17003290049365685 and parameters: {'learning_rate': 0.24519590546501618, 'max_depth': 6, 'n_estimators': 155, 'num_leaves': 66, 'subsample': 0.7410974347027424, 'colsample_bytree': 0.7603230799360103}. Best is trial 0 with value: 0.17003290049365685.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000287 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 09:00:10,695] Trial 1 finished with value: 0.17039174924072586 and parameters: {'learning_rate': 0.19173268507272972, 'max_depth': 3, 'n_estimators': 217, 'num_leaves': 76, 'subsample': 0.8601634323633273, 'colsample_bytree': 0.6594207547990049}. Best is trial 0 with value: 0.17003290049365685.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-01-06 09:00:26,159] Trial 48 finished with value: 0.1700092796942728 and parameters: {'learning_rate': 0.17255270226610728, 'max_depth': 6, 'n_estimators': 282, 'num_leaves': 52, 'subsample': 0.9773300062216268, 'colsample_bytree': 0.7423862913059978}. Best is trial 30 with value: 0.16997520728066537.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-06 09:00:26,621] Trial 49 finished with value: 0.1699964536474849 and parameters: {'learning_rate': 0.27135328953254995, 'max_depth': 9, 'n_estimators': 288, 'num_leaves': 85, 'subsample': 0.9742773125509481, 'colsample_bytree': 0.7165582199348524}. Best is trial 30 with value: 0.16997520728066537.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

#### bohb

In [ ]:
pip install hpbandster hyperopt


In [ ]:
!pip install --upgrade scikit-learn


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base models (XGBoost, CatBoost, LGB)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_max_depth", 3, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("xgb_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_iterations", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_depth", 4, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("cat_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_max_depth", -1, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lgb_learning_rate", 0.01, 0.3, default_value=0.1))

    # Final model (Decision Tree)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("dt_max_depth", 3, 15, default_value=6))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        # Base models with parameters from the configuration
        xgb = XGBRegressor(n_estimators=config["xgb_n_estimators"], max_depth=config["xgb_max_depth"], learning_rate=config["xgb_learning_rate"], random_state=42)
        cat = CatBoostRegressor(iterations=config["cat_iterations"], depth=config["cat_depth"], learning_rate=config["cat_learning_rate"], random_state=42, verbose=0)
        lgb = LGBMRegressor(n_estimators=config["lgb_n_estimators"], max_depth=config["lgb_max_depth"], learning_rate=config["lgb_learning_rate"], random_state=42)

        # Final model (Decision Tree)
        dt = DecisionTreeRegressor(max_depth=config["dt_max_depth"], random_state=42)

        # Stacked model
        stack_model = StackedRegressor(base_models=[xgb, cat, lgb], meta_model=dt)

        # Fit and evaluate on validation set
        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Set up BOHB
try:
    NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.0", port=None)
    NS.start()

    worker = StackingWorker(nameserver="127.0.0.0", run_id="stacking_bohb")
    worker.run(background=True)

    bohb = BOHB(
        configspace=get_config_space(),
        run_id="stacking_bohb",
        nameserver="127.0.0.0",
        min_budget=1,
        max_budget=3
    )

    # Perform optimization
    res = bohb.run(n_iterations=10)

    # Shutdown
    bohb.shutdown()
    NS.shutdown()

    # Retrieve the best configuration
    best_config = res.get_incumbent_id()
    best_params = res.get_id2config_mapping()[best_config]["config"]

    # Build the Stacked Ensemble model with the best hyperparameters
    best_xgb = XGBRegressor(n_estimators=best_params["xgb_n_estimators"], max_depth=best_params["xgb_max_depth"], learning_rate=best_params["xgb_learning_rate"], random_state=42)
    best_cat = CatBoostRegressor(iterations=best_params["cat_iterations"], depth=best_params["cat_depth"], learning_rate=best_params["cat_learning_rate"], random_state=42, verbose=0)
    best_lgb = LGBMRegressor(n_estimators=best_params["lgb_n_estimators"], max_depth=best_params["lgb_max_depth"], learning_rate=best_params["lgb_learning_rate"], random_state=42)
    best_dt = DecisionTreeRegressor(max_depth=best_params["dt_max_depth"], random_state=42)

    best_stack_model = StackedRegressor(base_models=[best_xgb, best_cat, best_lgb], meta_model=best_dt)

    # Fit the model
    best_stack_model.fit(X_train, Y_train)

    # Predict and evaluate
    Y_train_pred = best_stack_model.predict(X_train)
    Y_val_pred = best_stack_model.predict(X_val)
    Y_test_pred = best_stack_model.predict(X_test)

    # Performance metrics calculation
    train_metrics = calculate_metrics(Y_train, Y_train_pred)
    val_metrics = calculate_metrics(Y_val, Y_val_pred)
    test_metrics = calculate_metrics(Y_test, Y_test_pred)

    # Print the results
    print("Best Parameters Found by BOHB:")
    print(best_params)

    print("\nTraining set metrics:")
    print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

    print("\nValidation set metrics:")
    print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

    print("\nTest set metrics:")
    print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")

except Exception as e:
    print(f"Error occurred: {e}")
    if 'NS' in locals():
        NS.shutdown()  # Ensure NameServer is shut down on error


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000227 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 6188, number of used features: 3
[LightGBM] [Info] Start training from score 0.451818
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.base import RegressorMixin # Import RegressorMixin


# Wrapper classes for making models sklearn compatible
class XGBRegressorWrapper(XGBRegressor, RegressorMixin):
    pass

class CatBoostRegressorWrapper(CatBoostRegressor, RegressorMixin):
    def fit(self, X, y, **kwargs):
        return super().fit(X, y, verbose=0, **kwargs)

class LGBMRegressorWrapper(LGBMRegressor, RegressorMixin):
    pass

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base models (XGBoost, CatBoost, LGB)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_max_depth", 3, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("xgb_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_iterations", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_depth", 4, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("cat_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_max_depth", -1, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lgb_learning_rate", 0.01, 0.3, default_value=0.1))

    # Final model (Bagging Regressor)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("bag_n_estimators", 50, 300, default_value=100))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        # Base models with parameters from the configuration
        xgb = XGBRegressorWrapper(n_estimators=config["xgb_n_estimators"], max_depth=config["xgb_max_depth"], learning_rate=config["xgb_learning_rate"], random_state=42)
        cat = CatBoostRegressorWrapper(iterations=config["cat_iterations"], depth=config["cat_depth"], learning_rate=config["cat_learning_rate"], random_state=42, verbose=0)
        lgb = LGBMRegressorWrapper(n_estimators=config["lgb_n_estimators"], max_depth=config["lgb_max_depth"], learning_rate=config["lgb_learning_rate"], random_state=42)

        # Final model (Bagging)
        bag = BaggingRegressor(n_estimators=config["bag_n_estimators"], random_state=42)

        # Stacked model
        stack_model = StackingRegressor(estimators=[('xgb', xgb), ('cat', cat), ('lgb', lgb)], final_estimator=bag)

        # Fit and evaluate on validation set
        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Set up BOHB
try:
    NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.0", port=None)
    NS.start()

    worker = StackingWorker(nameserver="127.0.0.0", run_id="stacking_bohb")
    worker.run(background=True)

    bohb = BOHB(
        configspace=get_config_space(),
        run_id="stacking_bohb",
        nameserver="127.0.0.0",
        min_budget=1,
        max_budget=3
    )

    # Perform optimization
    res = bohb.run(n_iterations=10)

    # Shutdown
    bohb.shutdown()
    NS.shutdown()

    # Retrieve the best configuration
    best_config = res.get_incumbent_id()
    best_params = res.get_id2config_mapping()[best_config]["config"]

    # Build the Stacked Ensemble model with the best hyperparameters
    best_xgb = XGBRegressor(n_estimators=best_params["xgb_n_estimators"], max_depth=best_params["xgb_max_depth"], learning_rate=best_params["xgb_learning_rate"], random_state=42)
    best_cat = CatBoostRegressor(iterations=best_params["cat_iterations"], depth=best_params["cat_depth"], learning_rate=best_params["cat_learning_rate"], random_state=42, verbose=0)
    best_lgb = LGBMRegressor(n_estimators=best_params["lgb_n_estimators"], max_depth=best_params["lgb_max_depth"], learning_rate=best_params["lgb_learning_rate"], random_state=42)
    best_bag = BaggingRegressor(n_estimators=best_params["bag_n_estimators"], random_state=42)

    best_stack_model = StackingRegressor(estimators=[('xgb', best_xgb), ('cat', best_cat), ('lgb', best_lgb)], final_estimator=best_bag)

    # Fit the model
    best_stack_model.fit(X_train, Y_train)

    # Predict and evaluate
    Y_train_pred = best_stack_model.predict(X_train)
    Y_val_pred = best_stack_model.predict(X_val)
    Y_test_pred = best_stack_model.predict(X_test)

    # Performance metrics calculation
    train_metrics = calculate_metrics(Y_train, Y_train_pred)
    val_metrics = calculate_metrics(Y_val, Y_val_pred)
    test_metrics = calculate_metrics(Y_test, Y_test_pred)

    # Print the results
    print("Best Parameters Found by BOHB:")
    print(best_params)

    print("\nTraining set metrics:")
    print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

    print("\nValidation set metrics:")
    print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

    print("\nTest set metrics:")
    print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")

except Exception as e:
    print(f"Error occurred: {e}")
    if 'NS' in locals():
        NS.shutdown()  # Ensure NameServer is shut down on error


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000322 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000383 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 6188, number of used features: 3
[LightGBM] [Info] Start training from score 0.540058
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000229 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 6189, number of used features: 3
[LightGBM] [Info] Start training fro

### Base: XGBoost, CatBoost, LGB; Final: RandomForest

In [ ]:
!pip install catboost lightgbm xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 6.9 MB/s eta 0:00:00


#### initial

In [ ]:


import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define base models
xgb = XGBRegressor(random_state=42)
catboost = CatBoostRegressor(random_seed=42, silent=True)
lgb = LGBMRegressor(random_state=42)

# Train base models
xgb.fit(X_train, Y_train)
catboost.fit(X_train, Y_train)
lgb.fit(X_train, Y_train)

# Predict with base models
xgb_train_pred = xgb.predict(X_train)
catboost_train_pred = catboost.predict(X_train)
lgb_train_pred = lgb.predict(X_train)

xgb_val_pred = xgb.predict(X_val)
catboost_val_pred = catboost.predict(X_val)
lgb_val_pred = lgb.predict(X_val)

xgb_test_pred = xgb.predict(X_test)
catboost_test_pred = catboost.predict(X_test)
lgb_test_pred = lgb.predict(X_test)

# Stack predictions as new features
X_train_stack = np.column_stack((xgb_train_pred, catboost_train_pred, lgb_train_pred))
X_val_stack = np.column_stack((xgb_val_pred, catboost_val_pred, lgb_val_pred))
X_test_stack = np.column_stack((xgb_test_pred, catboost_test_pred, lgb_test_pred))

# Define final model
rf = RandomForestRegressor(random_state=42)

# Fit the final model on stacked features
rf.fit(X_train_stack, Y_train)

# Predict and evaluate
Y_train_pred = rf.predict(X_train_stack)
Y_val_pred = rf.predict(X_val_stack)
Y_test_pred = rf.predict(X_test_stack)

# Performance metrics calculation
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("Training set metrics:")
print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000839 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7736, number of used features: 3
[LightGBM] [Info] Start training from score 0.454038
Training set metrics:
MAE: 0.00126877509598725, MSE: 4.929985077275652e-06, RMSE: 0.002220356970686392, R²: 0.9999718320378708, MAPE: 0.0034649419351391302

Validation set metrics:
MAE: 0.1388144369162738, MSE: 0.02491673156020607, RMSE: 0.15785034545482018, R²: -3.236427365229636, MAPE: 0.07772546470877714

Test set metrics:
MAE: 0.4070239430443005, MSE: 0.17170356060232797, RMSE: 0.4143712835155544, R²: -27.450962371282667, MAPE: 0.20072143694912184


The metrics indicate strong performance on the training set, with near-perfect scores, suggesting the model fits the training data exceptionally well. However, the performance significantly drops on the validation and test sets, with higher errors and highly negative R² scores. This indicates poor generalization and overfitting, as the model fails to accurately predict unseen data.

#### optuna

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.8 MB/s eta 0:00:00


In [ ]:
!pip install optuna-integration[sklearn]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.4/97.4 kB 2.7 MB/s eta 0:00:00


In [ ]:
import optuna
import numpy as np
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error

# Stacked Regressor
class StackedRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_models=None, meta_model=None, n_folds=5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.n_folds = n_folds

    def fit(self, X, y):
        self.base_models_ = [list() for _ in self.base_models]
        self.meta_model_ = clone(self.meta_model)
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
        out_of_fold_predictions = np.zeros((X.shape[0], len(self.base_models)))

        for i, model in enumerate(self.base_models):
            for train_idx, holdout_idx in kfold.split(X, y):
                instance = clone(model)
                self.base_models_[i].append(instance)
                instance.fit(X.iloc[train_idx], y.iloc[train_idx])  # Ensure proper indexing
                y_pred = instance.predict(X.iloc[holdout_idx])  # Ensure proper indexing
                out_of_fold_predictions[holdout_idx, i] = y_pred

        self.meta_model_.fit(out_of_fold_predictions, y)
        return self

    def predict(self, X):
        meta_features = np.column_stack([
            np.column_stack([model.predict(X) for model in base_models]).mean(axis=1)
            for base_models in self.base_models_
        ])
        return self.meta_model_.predict(meta_features)

# Reset indices to avoid index misalignment
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
Y_train = Y_train.reset_index(drop=True)
Y_test = Y_test.reset_index(drop=True)

# Optuna Objective Function
def objective(trial):
    # XGBoost parameters
    xgb_params = {
        'n_estimators': trial.suggest_int('xgb_n_estimators', 50, 300),
        'max_depth': trial.suggest_int('xgb_max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('xgb_learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_uniform('xgb_subsample', 0.5, 1.0)
    }

    # CatBoost parameters
    cat_params = {
        'iterations': trial.suggest_int('cat_iterations', 50, 300),
        'depth': trial.suggest_int('cat_depth', 4, 10),
        'learning_rate': trial.suggest_loguniform('cat_learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_loguniform('cat_l2_leaf_reg', 1e-5, 10)
    }

    # LightGBM parameters
    lgb_params = {
        'n_estimators': trial.suggest_int('lgb_n_estimators', 50, 300),
        'max_depth': trial.suggest_int('lgb_max_depth', -1, 10),
        'learning_rate': trial.suggest_loguniform('lgb_learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_uniform('lgb_subsample', 0.5, 1.0)
    }

    # Base models
    xgb = XGBRegressor(**xgb_params, random_state=42)
    cat = CatBoostRegressor(**cat_params, random_state=42, verbose=0)
    lgb = LGBMRegressor(**lgb_params, random_state=42)

    # Final model (meta-model)
    rf_params = {
        'n_estimators': trial.suggest_int('rf_n_estimators', 100, 300),
        'max_depth': trial.suggest_int('rf_max_depth', 3, 15)
    }
    rf = RandomForestRegressor(**rf_params, random_state=42)

    stack_model = StackedRegressor(base_models=[xgb, cat, lgb], meta_model=rf)
    stack_model.fit(X_train, Y_train)
    y_pred = stack_model.predict(X_test)
    mse = mean_squared_error(Y_test, y_pred)

    return mse

# Run Optuna Study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)  # Reduced number of trials for faster execution

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


[I 2025-01-05 21:46:46,535] A new study created in memory with name: no-name-8f6f0c47-c5df-4970-87cd-2a5b92c60c60


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000264 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7072, number of used features: 3
[LightGBM] [Info] Start training from score 0.876999
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000447 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7073, number of used features: 3
[LightGBM] [Info] Start training from score 0.889001
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000367 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7073, number of used features: 3
[LightGBM] [Info] Start training fro

[I 2025-01-05 21:47:11,048] Trial 0 finished with value: 2.7319363495525294e-06 and parameters: {'xgb_n_estimators': 285, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.1626698185738664, 'xgb_subsample': 0.7497367799099663, 'cat_iterations': 84, 'cat_depth': 6, 'cat_learning_rate': 0.0305088032351903, 'cat_l2_leaf_reg': 0.0018787937556708513, 'lgb_n_estimators': 225, 'lgb_max_depth': 5, 'lgb_learning_rate': 0.013586059150182092, 'lgb_subsample': 0.8553162083313284, 'rf_n_estimators': 202, 'rf_max_depth': 11}. Best is trial 0 with value: 2.7319363495525294e-06.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000274 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7072, number of used features: 3
[LightGBM] [Info] Start training from score 0.876999
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7073, number of used features: 3
[LightGBM] [Info] Start training from score 0.889001
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000276 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7073, number of used features: 3
[LightGBM] [Info] Start training fro

[I 2025-01-05 21:47:34,678] Trial 1 finished with value: 0.00013510420423994653 and parameters: {'xgb_n_estimators': 293, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.07116497078472792, 'xgb_subsample': 0.5497843213039066, 'cat_iterations': 245, 'cat_depth': 10, 'cat_learning_rate': 0.02513847539142229, 'cat_l2_leaf_reg': 0.8175793424357376, 'lgb_n_estimators': 143, 'lgb_max_depth': 5, 'lgb_learning_rate': 0.025629867135060196, 'lgb_subsample': 0.8537929269714994, 'rf_n_estimators': 131, 'rf_max_depth': 5}. Best is trial 0 with value: 2.7319363495525294e-06.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000259 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7072, number of used features: 3
[LightGBM] [Info] Start training from score 0.876999
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-05 21:47:56,490] Trial 3 finished with value: 3.50675990815661e-06 and parameters: {'xgb_n_estimators': 173, 'xgb_max_depth': 5, 'xgb_learning_rate': 0.10919828147773576, 'xgb_subsample': 0.7619183166917505, 'cat_iterations': 93, 'cat_depth': 8, 'cat_learning_rate': 0.011552713845657848, 'cat_l2_leaf_reg': 1.3258721527992998, 'lgb_n_estimators': 221, 'lgb_max_depth': 1, 'lgb_learning_rate': 0.018565524238300823, 'lgb_subsample': 0.5103036268548093, 'rf_n_estimators': 148, 'rf_max_depth': 14}. Best is trial 0 with value: 2.7319363495525294e-06.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000283 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7072, number of used features: 3
[LightGBM] [Info] Start training from score 0.876999
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-01-05 21:48:08,574] Trial 4 finished with value: 5.808876409663977e-06 and parameters: {'xgb_n_estimators': 211, 'xgb_max_depth': 7, 'xgb_learning_rate': 0.14095301551087083, 'xgb_subsample': 0.9124182704774467, 'cat_iterations': 78, 'cat_depth': 10, 'cat_learning_rate': 0.15592803391438564, 'cat_l2_leaf_reg': 0.002287969454403588, 'lgb_n_estimators': 239, 'lgb_max_depth': 2, 'lgb_learning_rate': 0.011369628752624627, 'lgb_subsample': 0.708597286342868, 'rf_n_estimators': 163, 'rf_max_depth': 7}. Best is trial 0 with value: 2.7319363495525294e-06.


Best trial:
  Value:  2.7319363495525294e-06
  Params: 
    xgb_n_estimators: 285
    xgb_max_depth: 6
    xgb_learning_rate: 0.1626698185738664
    xgb_subsample: 0.7497367799099663
    cat_iterations: 84
    cat_depth: 6
    cat_learning_rate: 0.0305088032351903
    cat_l2_leaf_reg: 0.0018787937556708513
    lgb_n_estimators: 225
    lgb_max_depth: 5
    lgb_learning_rate: 0.013586059150182092
    lgb_subsample: 0.8553162083313284
    rf_n_estimators: 202
    rf_max_depth: 11


In [ ]:
# Best parameters from the optimized trial
best_params = trial.params

# Set up the models with the best parameters
xgb = XGBRegressor(
    n_estimators=best_params['xgb_n_estimators'],
    max_depth=best_params['xgb_max_depth'],
    learning_rate=best_params['xgb_learning_rate'],
    subsample=best_params['xgb_subsample'],
    random_state=42
)

cat = CatBoostRegressor(
    iterations=best_params['cat_iterations'],
    depth=best_params['cat_depth'],
    learning_rate=best_params['cat_learning_rate'],
    l2_leaf_reg=best_params['cat_l2_leaf_reg'],
    random_state=42,
    verbose=0
)

lgb = LGBMRegressor(
    n_estimators=best_params['lgb_n_estimators'],
    max_depth=best_params['lgb_max_depth'],
    learning_rate=best_params['lgb_learning_rate'],
    subsample=best_params['lgb_subsample'],
    random_state=42
)

rf = RandomForestRegressor(
    n_estimators=best_params['rf_n_estimators'],
    max_depth=best_params['rf_max_depth'],
    random_state=42
)

# Create the stacked model
stack_model = StackedRegressor(base_models=[xgb, cat, lgb], meta_model=rf)

# Fit the model on the entire training set
stack_model.fit(X_train, Y_train)

# Predict on train, validation, and test sets
train_pred = stack_model.predict(X_train)
val_pred = stack_model.predict(X_val)
test_pred = stack_model.predict(X_test)

# Calculate the metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, mse, rmse, r2, mape

# Train metrics
train_metrics = calculate_metrics(Y_train, train_pred)

# Validation metrics
val_metrics = calculate_metrics(Y_val, val_pred)

# Test metrics
test_metrics = calculate_metrics(Y_test, test_pred)

# Print results
print("Train Metrics: MAE, MSE, RMSE, R², MAPE")
print(train_metrics)

print("\nValidation Metrics: MAE, MSE, RMSE, R², MAPE")
print(val_metrics)

print("\nTest Metrics: MAE, MSE, RMSE, R², MAPE")
print(test_metrics)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000269 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7072, number of used features: 3
[LightGBM] [Info] Start training from score 0.876999
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000269 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7073, number of used features: 3
[LightGBM] [Info] Start training from score 0.889001
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000265 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7073, number of used features: 3
[LightGBM] [Info] Start training fro

 shows excellent generalization, with very low error metrics across all datasets, making it a robust model for the problem at hand. The extremely high R² values indicate that the model is capturing almost all of the variance in the data.



In [ ]:
!pip install --upgrade scikit-learn

#### bohb

In [ ]:
!pip install hpbandster
!pip install ConfigSpace

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base models (XGBoost, CatBoost, LGB)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("xgb_max_depth", 3, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("xgb_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_iterations", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("cat_depth", 4, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("cat_learning_rate", 0.01, 0.3, default_value=0.1))

    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("lgb_max_depth", -1, 10, default_value=6))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lgb_learning_rate", 0.01, 0.3, default_value=0.1))

    # Final model (Random Forest)
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("rf_n_estimators", 50, 300, default_value=100))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("rf_max_depth", 3, 15, default_value=6))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        # Base models with parameters from the configuration
        xgb = XGBRegressor(n_estimators=config["xgb_n_estimators"], max_depth=config["xgb_max_depth"], learning_rate=config["xgb_learning_rate"], random_state=42)
        cat = CatBoostRegressor(iterations=config["cat_iterations"], depth=config["cat_depth"], learning_rate=config["cat_learning_rate"], random_state=42, verbose=0)
        lgb = LGBMRegressor(n_estimators=config["lgb_n_estimators"], max_depth=config["lgb_max_depth"], learning_rate=config["lgb_learning_rate"], random_state=42)

        # Final model (Random Forest)
        rf = RandomForestRegressor(n_estimators=config["rf_n_estimators"], max_depth=config["rf_max_depth"], random_state=42)

        # Stacked model
        stack_model = StackedRegressor(base_models=[xgb, cat, lgb], meta_model=rf)

        # Fit and evaluate on validation set
        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Set up BOHB
try:
    NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.0", port=None)
    NS.start()

    worker = StackingWorker(nameserver="127.0.0.0", run_id="stacking_bohb")
    worker.run(background=True)

    bohb = BOHB(
        configspace=get_config_space(),
        run_id="stacking_bohb",
        nameserver="127.0.0.0",
        min_budget=1,
        max_budget=3
    )

    # Perform optimization
    res = bohb.run(n_iterations=10)

    # Shutdown
    bohb.shutdown()
    NS.shutdown()

    # Retrieve the best configuration
    best_config = res.get_incumbent_id()
    best_params = res.get_id2config_mapping()[best_config]["config"]

    # Build the Stacked Ensemble model with the best hyperparameters
    best_xgb = XGBRegressor(n_estimators=best_params["xgb_n_estimators"], max_depth=best_params["xgb_max_depth"], learning_rate=best_params["xgb_learning_rate"], random_state=42)
    best_cat = CatBoostRegressor(iterations=best_params["cat_iterations"], depth=best_params["cat_depth"], learning_rate=best_params["cat_learning_rate"], random_state=42, verbose=0)
    best_lgb = LGBMRegressor(n_estimators=best_params["lgb_n_estimators"], max_depth=best_params["lgb_max_depth"], learning_rate=best_params["lgb_learning_rate"], random_state=42)
    best_rf = RandomForestRegressor(n_estimators=best_params["rf_n_estimators"], max_depth=best_params["rf_max_depth"], random_state=42)

    best_stack_model = StackedRegressor(base_models=[best_xgb, best_cat, best_lgb], meta_model=best_rf)

    # Fit the model
    best_stack_model.fit(X_train, Y_train)

    # Predict and evaluate
    Y_train_pred = best_stack_model.predict(X_train)
    Y_val_pred = best_stack_model.predict(X_val)
    Y_test_pred = best_stack_model.predict(X_test)

    # Performance metrics calculation
    train_metrics = calculate_metrics(Y_train, Y_train_pred)
    val_metrics = calculate_metrics(Y_val, Y_val_pred)
    test_metrics = calculate_metrics(Y_test, Y_test_pred)

    # Print the results
    print("Best Parameters Found by BOHB:")
    print(best_params)

    print("\nTraining set metrics:")
    print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

    print("\nValidation set metrics:")
    print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

    print("\nTest set metrics:")
    print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")

except Exception as e:
    print(f"Error occurred: {e}")
    if 'NS' in locals():
        NS.shutdown()  # Ensure NameServer is shut down on error


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000422 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 7072, number of used features: 3
[LightGBM] [Info] Start training from score 0.876999
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

### Base: Ridge, SVR, Linear; Final: Lasso

#### Initial

In [ ]:
import time
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define base models
base_models = [
    ('linear', LinearRegression()),
    ('ridge', Ridge()),
    ('svr', SVR())
]

# Define final model (Lasso)
final_model = Lasso()

# Create the Stacking Regressor
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=final_model
)

# Standardize data (important for SVR)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Train the stacking model
start_train_time = time.time()
stacking_model.fit(X_train_scaled, Y_train)
train_time = time.time() - start_train_time

# Make predictions
final_preds_train = stacking_model.predict(X_train_scaled)
final_preds_val = stacking_model.predict(X_val_scaled)
final_preds_test = stacking_model.predict(X_test_scaled)

# Evaluate performance (train, validation, and test)
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return rmse, mse, mae, r2, mape

# Training performance
train_rmse, train_mse, train_mae, train_r2, train_mape = evaluate(Y_train, final_preds_train)
# Validation performance
val_rmse, val_mse, val_mae, val_r2, val_mape = evaluate(Y_val, final_preds_val)
# Test performance
test_rmse, test_mse, test_mae, test_r2, test_mape = evaluate(Y_test, final_preds_test)

# Print results
print("Training Results:")
print(f"RMSE: {train_rmse}, MSE: {train_mse}, MAE: {train_mae}, R²: {train_r2}, MAPE: {train_mape}")

print("\nValidation Results:")
print(f"RMSE: {val_rmse}, MSE: {val_mse}, MAE: {val_mae}, R²: {val_r2}, MAPE: {val_mape}")

print("\nTest Results:")
print(f"RMSE: {test_rmse}, MSE: {test_mse}, MAE: {test_mae}, R²: {test_r2}, MAPE: {test_mape}")

print(f"\nTime taken for training the stacking model: {train_time:.4f} seconds")


Training Results:
RMSE: 0.4183551198443149, MSE: 0.17502100629995107, MAE: 0.3278230129142329, R²: 0.0, MAPE: 1.2517838355180178

Validation Results:
RMSE: 1.2947378257698354, MSE: 1.676346037479201, MAE: 1.292464503978209, R²: -284.0180494023197, MAPE: 0.7395216940010845

Test Results:
RMSE: 1.5634328530455037, MSE: 2.4443222859820035, MAE: 1.5615015900059166, R²: -404.01968123320756, MAPE: 0.7743863775989874

Time taken for training the stacking model: 0.1757 seconds


The Stacked Ensemble model shows poor performance, especially on the validation and test sets, with significantly high error values and negative R² scores. This indicates overfitting or the base models and final model (Lasso) may not be appropriate for this dataset or task. Further tuning, alternative models, or additional feature engineering may be necessary to improve the model's performance.

#### optuna

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.6 MB/s eta 0:00:00


In [ ]:
import optuna
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
import numpy as np

# Standardize data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Define the objective function for each model
def tune_ridge(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-5, 1e2)
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, Y_train)
    preds = model.predict(X_val_scaled)
    return mean_squared_error(Y_val, preds)

def tune_svr(trial):
    C = trial.suggest_loguniform('C', 1e-5, 1e2)
    epsilon = trial.suggest_loguniform('epsilon', 1e-5, 1e1)
    model = SVR(C=C, epsilon=epsilon)
    model.fit(X_train_scaled, Y_train)
    preds = model.predict(X_val_scaled)
    return mean_squared_error(Y_val, preds)

def tune_lasso(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-5, 1e2)
    model = Lasso(alpha=alpha)
    model.fit(X_train_scaled, Y_train)
    preds = model.predict(X_val_scaled)
    return mean_squared_error(Y_val, preds)

# Optimize each model
ridge_study = optuna.create_study(direction='minimize')
ridge_study.optimize(tune_ridge, n_trials=50)
ridge_best_params = ridge_study.best_params

svr_study = optuna.create_study(direction='minimize')
svr_study.optimize(tune_svr, n_trials=50)
svr_best_params = svr_study.best_params

lasso_study = optuna.create_study(direction='minimize')
lasso_study.optimize(tune_lasso, n_trials=50)
lasso_best_params = lasso_study.best_params

# Initialize tuned models
ridge_tuned = Ridge(**ridge_best_params)
svr_tuned = SVR(**svr_best_params)
lasso_tuned = Lasso(**lasso_best_params)
linear_model = LinearRegression()

# Define base models for StackingRegressor
base_models = [
    ('linear', linear_model),
    ('ridge', ridge_tuned),
    ('svr', svr_tuned)
]

# Final estimator (meta-model)
final_model = lasso_tuned

# Create and train StackingRegressor
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=final_model
)

stacking_model.fit(X_train_scaled, Y_train)

# Make predictions with the stacking model
final_preds_train = stacking_model.predict(X_train_scaled)
final_preds_val = stacking_model.predict(X_val_scaled)
final_preds_test = stacking_model.predict(X_test_scaled)

# Evaluate performance
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return rmse, mse, mae, r2, mape

# Training performance
train_rmse, train_mse, train_mae, train_r2, train_mape = evaluate(Y_train, final_preds_train)
# Validation performance
val_rmse, val_mse, val_mae, val_r2, val_mape = evaluate(Y_val, final_preds_val)
# Test performance
test_rmse, test_mse, test_mae, test_r2, test_mape = evaluate(Y_test, final_preds_test)

# Print results
print("Training Results:")
print(f"RMSE: {train_rmse}, MSE: {train_mse}, MAE: {train_mae}, R²: {train_r2}, MAPE: {train_mape}")

print("\nValidation Results:")
print(f"RMSE: {val_rmse}, MSE: {val_mse}, MAE: {val_mae}, R²: {val_r2}, MAPE: {val_mape}")

print("\nTest Results:")
print(f"RMSE: {test_rmse}, MSE: {test_mse}, MAE: {test_mae}, R²: {test_r2}, MAPE: {test_mape}")


[I 2025-01-04 19:12:29,042] A new study created in memory with name: no-name-740532a3-2fde-4264-99bb-2b7cc6e9049f
[I 2025-01-04 19:12:29,062] Trial 0 finished with value: 1.6667126494371767e-06 and parameters: {'alpha': 0.0007329092500138463}. Best is trial 0 with value: 1.6667126494371767e-06.
[I 2025-01-04 19:12:29,082] Trial 1 finished with value: 1.6693368334852228e-06 and parameters: {'alpha': 0.0036970599053382106}. Best is trial 0 with value: 1.6667126494371767e-06.
[I 2025-01-04 19:12:29,100] Trial 2 finished with value: 1.666176030742439e-06 and parameters: {'alpha': 9.268950658954587e-05}. Best is trial 2 with value: 1.666176030742439e-06.
[I 2025-01-04 19:12:29,129] Trial 3 finished with value: 1.7028849309018796e-06 and parameters: {'alpha': 0.031119763284731813}. Best is trial 2 with value: 1.666176030742439e-06.
[I 2025-01-04 19:12:29,139] Trial 4 finished with value: 1.6669460698459613e-06 and parameters: {'alpha': 0.001007420755919519}. Best is trial 2 with value: 1.666

Training Results:
RMSE: 0.002877899167432581, MSE: 8.282303617909143e-06, MAE: 0.0018912482294574946, R²: 0.9999526782310707, MAPE: 0.005826274055006865

Validation Results:
RMSE: 0.0013406429349951453, MSE: 1.7973234791523976e-06, MAE: 0.001003834174309906, R²: 0.9996944129548913, MAPE: 0.0005810374284019038

Test Results:
RMSE: 0.0011182503762865357, MSE: 1.2504839040649788e-06, MAE: 0.0007905328132859978, R²: 0.9997927971302655, MAPE: 0.00039579010170994855


Key Observations:
High R² Scores: The model explains almost all the variability in the target variable.
Low Error Metrics: Very low RMSE, MSE, and MAE indicate excellent prediction accuracy.
Low MAPE: The percentage errors are minimal, showing highly precise predictions.
Consistency Across Datasets: The performance is consistent across training, validation, and test sets, indicating good generalization.

#### bohb

In [ ]:
!pip install hpbandster

In [ ]:
import numpy as np
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base learners (Ridge, SVR, and Linear Regression)
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("ridge_alpha", 0.1, 10.0, default_value=1.0))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("svr_C", 0.1, 10.0, default_value=1.0))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("svr_epsilon", 0.01, 1.0, default_value=0.1))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("linear_fit_intercept", 0, 1, default_value=1))

    # Final model (Lasso)
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lasso_alpha", 0.1, 10.0, default_value=1.0))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        # Define base models with parameters from the configuration
        ridge = Ridge(alpha=config["ridge_alpha"])
        svr = SVR(C=config["svr_C"], epsilon=config["svr_epsilon"])
        linear = LinearRegression(fit_intercept=bool(config["linear_fit_intercept"]))

        # Define final model (Lasso)
        lasso = Lasso(alpha=config["lasso_alpha"])

        # Define Stacking Regressor
        stack_model = StackingRegressor(
            estimators=[("ridge", ridge), ("svr", svr), ("linear", linear)],
            final_estimator=lasso
        )

        # Fit and evaluate on validation set
        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Set up BOHB
NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.1", port=None)
NS.start()

worker = StackingWorker(nameserver="127.0.0.1", run_id="stacking_bohb")
worker.run(background=True)

bohb = BOHB(
    configspace=get_config_space(),
    run_id="stacking_bohb",
    nameserver="127.0.0.1",
    min_budget=1,
    max_budget=3
)

# Perform optimization
res = bohb.run(n_iterations=50)

# Shutdown
bohb.shutdown()
NS.shutdown()

# Retrieve the best configuration
best_config = res.get_incumbent_id()
best_params = res.get_id2config_mapping()[best_config]["config"]

# Build the Stacked Ensemble model with the best hyperparameters
best_ridge = Ridge(alpha=best_params["ridge_alpha"])
best_svr = SVR(C=best_params["svr_C"], epsilon=best_params["svr_epsilon"])
best_linear = LinearRegression(fit_intercept=bool(best_params["linear_fit_intercept"]))
best_lasso = Lasso(alpha=best_params["lasso_alpha"])

best_stack_model = StackingRegressor(
    estimators=[("ridge", best_ridge), ("svr", best_svr), ("linear", best_linear)],
    final_estimator=best_lasso
)

# Fit the model
best_stack_model.fit(X_train, Y_train)

# Predict and evaluate
Y_train_pred = best_stack_model.predict(X_train)
Y_val_pred = best_stack_model.predict(X_val)
Y_test_pred = best_stack_model.predict(X_test)

# Performance metrics calculation
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("Best Parameters Found by BOHB:")
print(best_params)

print("\nTraining set metrics:")
print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")


Best Parameters Found by BOHB:
{'lasso_alpha': 0.1052894677226, 'linear_fit_intercept': 0.9860259124655, 'ridge_alpha': 4.1571410558209, 'svr_C': 8.8420649411856, 'svr_epsilon': 0.7840005235296}

Training set metrics:
MAE: 0.19722570097774447, MSE: 0.06336246382150243, RMSE: 0.2517190175999867, R²: 0.6379722345275982, MAPE: 0.7531511392118198

Validation set metrics:
MAE: 0.7775651654607182, MSE: 0.6067388763161964, RMSE: 0.7789344493063562, R²: -102.15980540881806, MAPE: 0.444906209906654

Test set metrics:
MAE: 0.9394141285766106, MSE: 0.8846846599580184, RMSE: 0.9405767698375388, R²: -145.5906116484769, MAPE: 0.4658778950133781


The model performs well on the training set with low errors but a 0 R², indicating potential underfitting. On the validation and test sets, performance drops significantly, with high errors and negative R², suggesting poor generalization and overfitting.

The Optuna-tuned model was the best performer among the three, providing balanced and robust results across training, validation, and test sets. The BOHB-tuned model did not achieve the expected improvement, underperforming compared to the Optuna approach.

### Base: Ridge, KNN, Linear; Final: Lasso

#### initial

In [ ]:
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Base models
ridge = Ridge(alpha=1.0)
knn = KNeighborsRegressor(n_neighbors=5)
linear = LinearRegression()

# Final model (meta-model)
lasso = Lasso(alpha=0.1)

# Stacking Regressor
stack_model = StackingRegressor(
    estimators=[('ridge', ridge), ('knn', knn), ('linear', linear)],
    final_estimator=lasso
)

# Splitting the data (assuming X_train, Y_train, X_val, Y_val are already prepared)
stack_model.fit(X_train, Y_train)

# Predictions
Y_train_pred = stack_model.predict(X_train)
Y_val_pred = stack_model.predict(X_val)
Y_test_pred = stack_model.predict(X_test)

# Calculate metrics
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print results
print("\nTraining set metrics:")
print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")



Training set metrics:
MAE: 0.1873119607175348, MSE: 0.05715431755987886, RMSE: 0.23906969184712407, R²: 0.6734430982420032, MAPE: 0.7152992678330113

Validation set metrics:
MAE: 0.7384780688130521, MSE: 0.5472725978671682, RMSE: 0.739778749267082, R²: -92.04914668453452, MAPE: 0.42254132605604716

Test set metrics:
MAE: 0.8921901550554975, MSE: 0.7979749936877708, RMSE: 0.8932944607954147, R²: -131.22298034469227, MAPE: 0.44245836429467833


#### optuna

In [ ]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import StackingRegressor
import numpy as np

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define the objective function for Optuna optimization
def objective(trial):
    # Hyperparameters for each base model
    ridge_alpha = trial.suggest_loguniform('ridge_alpha', 1e-5, 100)
    knn_n_neighbors = trial.suggest_int('knn_n_neighbors', 1, 30)
    knn_weights = trial.suggest_categorical('knn_weights', ['uniform', 'distance'])
    linear_fit_intercept = trial.suggest_categorical('linear_fit_intercept', [True, False])

    # Define base models with the suggested hyperparameters
    ridge = Ridge(alpha=ridge_alpha)
    knn = KNeighborsRegressor(n_neighbors=knn_n_neighbors, weights=knn_weights)
    linear = LinearRegression(fit_intercept=linear_fit_intercept)

    # Final model (Lasso)
    lasso_alpha = trial.suggest_loguniform('lasso_alpha', 1e-5, 100)
    lasso = Lasso(alpha=lasso_alpha)

    # Define Stacking Regressor
    stack_model = StackingRegressor(
        estimators=[('ridge', ridge), ('knn', knn), ('linear', linear)],
        final_estimator=lasso
    )

    # Train the model
    stack_model.fit(X_train, Y_train)

    # Predict on validation set
    Y_val_pred = stack_model.predict(X_val)

    # Calculate MAE for validation set
    mae = mean_absolute_error(Y_val, Y_val_pred)
    return mae

# Run Optuna optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Get the best parameters and build the final model
best_params = study.best_params
print(f"Best Parameters: {best_params}")

# Create and train the final model with the best parameters
ridge = Ridge(alpha=best_params['ridge_alpha'])
knn = KNeighborsRegressor(n_neighbors=best_params['knn_n_neighbors'], weights=best_params['knn_weights'])
linear = LinearRegression(fit_intercept=best_params['linear_fit_intercept'])
lasso = Lasso(alpha=best_params['lasso_alpha'])

stack_model = StackingRegressor(
    estimators=[('ridge', ridge), ('knn', knn), ('linear', linear)],
    final_estimator=lasso
)

# Fit the final model
stack_model.fit(X_train, Y_train)

# Predict and evaluate on each dataset
Y_train_pred = stack_model.predict(X_train)
Y_val_pred = stack_model.predict(X_val)
Y_test_pred = stack_model.predict(X_test)

train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print the results
print("\nTraining set metrics:")
print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")


[I 2025-01-04 20:18:51,834] A new study created in memory with name: no-name-43b145f0-2f74-4b73-bf3e-66a8e2b480d5
[I 2025-01-04 20:18:52,149] Trial 0 finished with value: 0.029444868002810477 and parameters: {'ridge_alpha': 10.589935507436298, 'knn_n_neighbors': 14, 'knn_weights': 'distance', 'linear_fit_intercept': True, 'lasso_alpha': 0.00039647708469936965}. Best is trial 0 with value: 0.029444868002810477.
[I 2025-01-04 20:18:52,407] Trial 1 finished with value: 1.292464503978209 and parameters: {'ridge_alpha': 0.00011472173493732895, 'knn_n_neighbors': 5, 'knn_weights': 'distance', 'linear_fit_intercept': True, 'lasso_alpha': 14.303290485978533}. Best is trial 0 with value: 0.029444868002810477.
[I 2025-01-04 20:18:52,629] Trial 2 finished with value: 0.021837444399612098 and parameters: {'ridge_alpha': 0.1666213778422776, 'knn_n_neighbors': 13, 'knn_weights': 'uniform', 'linear_fit_intercept': False, 'lasso_alpha': 0.0030381840301649267}. Best is trial 2 with value: 0.02183744439

Best Parameters: {'ridge_alpha': 0.00019516344788784412, 'knn_n_neighbors': 3, 'knn_weights': 'distance', 'linear_fit_intercept': False, 'lasso_alpha': 5.6199717205259336e-05}

Training set metrics:
MAE: 0.0018918380143607787, MSE: 8.277793130339302e-06, RMSE: 0.002877115418320805, R²: 0.9999527040021918, MAPE: 0.005845560133693613

Validation set metrics:
MAE: 0.0009550796990908053, MSE: 1.655481199784465e-06, RMSE: 0.0012866550430416324, R²: 0.9997185294611999, MAPE: 0.0005531849732641942

Test set metrics:
MAE: 0.0006926112060595493, MSE: 1.0192895215060405e-06, RMSE: 0.0010095986932965198, R²: 0.9998311056117877, MAPE: 0.0003476829529759195


#### bohb

In [ ]:

import hpbandster.core.nameserver as hpns
from hpbandster.optimizers import BOHB
from hpbandster.core.worker import Worker
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import numpy as np

# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mae, mse, rmse, r2, mape

# Define configuration space for hyperparameter optimization
def get_config_space():
    cs = CS.ConfigurationSpace()

    # Base learners
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("ridge_alpha", 0.1, 10.0, default_value=1.0))
    cs.add_hyperparameter(CSH.UniformIntegerHyperparameter("knn_n_neighbors", 1, 50, default_value=5))
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("linear_fit_intercept", 0, 1, default_value=1))

    # Final model (Lasso)
    cs.add_hyperparameter(CSH.UniformFloatHyperparameter("lasso_alpha", 0.1, 10.0, default_value=1.0))

    return cs

# Define worker for BOHB
class StackingWorker(Worker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def compute(self, config, budget, **kwargs):
        ridge = Ridge(alpha=config["ridge_alpha"])
        knn = KNeighborsRegressor(n_neighbors=config["knn_n_neighbors"])
        linear = LinearRegression(fit_intercept=bool(config["linear_fit_intercept"]))

        # Final estimator
        lasso = Lasso(alpha=config["lasso_alpha"])

        stack_model = StackingRegressor(
            estimators=[("ridge", ridge), ("knn", knn), ("linear", linear)],
            final_estimator=lasso
        )

        stack_model.fit(X_train, Y_train)
        Y_val_pred = stack_model.predict(X_val)
        mae = mean_absolute_error(Y_val, Y_val_pred)

        return {"loss": mae, "info": config}

# Start the Name Server
NS = hpns.NameServer(run_id="stacking_bohb", host="127.0.0.1", port=None)
NS.start()

worker = StackingWorker(nameserver="127.0.0.1", run_id="stacking_bohb")
worker.run(background=True)

bohb = BOHB(
    configspace=get_config_space(),
    run_id="stacking_bohb",
    nameserver="127.0.0.1",
    min_budget=1,
    max_budget=3
)

# Perform optimization
res = bohb.run(n_iterations=50)

# Shutdown
bohb.shutdown()
NS.shutdown()

# Retrieve the best configuration
best_config = res.get_incumbent_id()
best_params = res.get_id2config_mapping()[best_config]["config"]

# Build the Stacked Ensemble model with the best hyperparameters
best_ridge = Ridge(alpha=best_params["ridge_alpha"])
best_knn = KNeighborsRegressor(n_neighbors=best_params["knn_n_neighbors"])
best_linear = LinearRegression(fit_intercept=bool(best_params["linear_fit_intercept"]))
best_lasso = Lasso(alpha=best_params["lasso_alpha"])

best_stack_model = StackingRegressor(
    estimators=[("ridge", best_ridge), ("knn", best_knn), ("linear", best_linear)],
    final_estimator=best_lasso
)

# Fit and predict
best_stack_model.fit(X_train, Y_train)
Y_train_pred = best_stack_model.predict(X_train)
Y_val_pred = best_stack_model.predict(X_val)
Y_test_pred = best_stack_model.predict(X_test)

# Calculate metrics
train_metrics = calculate_metrics(Y_train, Y_train_pred)
val_metrics = calculate_metrics(Y_val, Y_val_pred)
test_metrics = calculate_metrics(Y_test, Y_test_pred)

# Print results
print("Best Parameters Found by BOHB:")
print(best_params)

print("\nTraining set metrics:")
print(f"MAE: {train_metrics[0]}, MSE: {train_metrics[1]}, RMSE: {train_metrics[2]}, R²: {train_metrics[3]}, MAPE: {train_metrics[4]}")

print("\nValidation set metrics:")
print(f"MAE: {val_metrics[0]}, MSE: {val_metrics[1]}, RMSE: {val_metrics[2]}, R²: {val_metrics[3]}, MAPE: {val_metrics[4]}")

print("\nTest set metrics:")
print(f"MAE: {test_metrics[0]}, MSE: {test_metrics[1]}, RMSE: {test_metrics[2]}, R²: {test_metrics[3]}, MAPE: {test_metrics[4]}")


Best Parameters Found by BOHB:
{'knn_n_neighbors': 18, 'lasso_alpha': 8.8512739390074, 'linear_fit_intercept': 0.0115351684671, 'ridge_alpha': 8.4167000380091}

Training set metrics:
MAE: 0.3278230129142329, MSE: 0.17502100629995107, RMSE: 0.4183551198443149, R²: 0.0, MAPE: 1.2517838355180178

Validation set metrics:
MAE: 1.292464503978209, MSE: 1.676346037479201, RMSE: 1.2947378257698354, R²: -284.0180494023197, MAPE: 0.7395216940010845

Test set metrics:
MAE: 1.5615015900059166, MSE: 2.4443222859820035, RMSE: 1.5634328530455037, R²: -404.01968123320756, MAPE: 0.7743863775989874


Overfitting: The model performs well on the training data but struggles to generalize to unseen data, likely due to overfitting. This is supported by the sharp decline in performance on the validation and test sets.
Poor Generalization: The negative R² and high error metrics on the test/validation sets show that the model isn't capturing the underlying patterns in the data effectively.